# Cascade M-Protein CDS — Training Pipeline

In [ ]:
"""Cascade M-Protein CDS — Training Pipeline

Automatically generated by Colab.

# Cascade M-Protein CDS — Model Training & Evaluation Pipeline

**L1 (Binary) → L2 (Heavy Chain) → L3 (Light Chain) → Cascade → CDS**

End-to-end pipeline for training, evaluating, and validating a 3-level
hierarchical cascade classifier for M-protein isotype classification from
6-channel capillary zone electrophoresis with immunotyping (CZE-IT) signals.

**Outputs:** All OOF predictions, trained models, calibration results,
conformal prediction, and cascade metrics saved as `.pkl` files.

## Table of Contents

| § | Section | Description |
|---|---------|-------------|
| — | Data Preparation (Optional) | Build dataset.pkl from Excel/TSV — skip if pkl exists |
| 0 | Setup & Configuration | Paths, imports, config presets, data loading, fold assignments |
| 1 | Peak Feature Extraction | 399 peak-based features from 6-channel CZE-IT signals |
| 2 | Flat Baselines (9-Class) | Logistic Regression & XGBoost flat 9-class classifiers |
| 3 | Level 1 — Binary Detection | 7 models + Optuna + threshold optimization + ensemble |
| 4 | Level 2 — Heavy Chain | 4 models + Optuna (IgG / IgA / IgM / Free) |
| 5 | Level 3 — Light Chain | 4 models + Optuna (κ vs λ) |
| 6 | Cascade Assembly | L1→L2→L3 → 9-class evaluation + flat comparison |
| 7 | Bootstrap Confidence Intervals | 1,000 bootstrap iterations for cascade metrics |
| 8 | Calibration Analysis | ECE, MCE, BSS per level + isotonic sensitivity check |
| 9 | Conformal Prediction & Confidence Zones | Split conformal + compound confidence + triaging |
| 10 | Error Attribution | Per-error source decomposition (L1-FN/FP, L2, L3) |
| 11 | External Validation | Frozen weights on independent cohort (n=498) |
| 12 | SHAP Explainability (500 samples) | Region × channel heatmap |
| 13 | CDS Pipeline Demo | Single-sample end-to-end CDS inference |
| 14 | Save All Results | Cascade, calibration, conformal, confidence pkl files |
| 15 | Learning Curves | L1/L2/L3 sample efficiency (10 subsample points × 5-fold) |
| 16 | Tuning-Free Cascade | Default XGBoost cascade — hyperparameter overfitting check |
| 17 | Nested CV — Optuna ⚠️ OPTIONAL | 5 outer × inner Optuna — gold-standard evaluation |
| 18 | Full SHAP Computation | All samples × all levels × both cohorts (internal + external) |
"""

══════════════════════════════════════════════════════════════
DATA PREPARATION (Optional — skip if dataset.pkl already exists)
══════════════════════════════════════════════════════════════

## Data Preparation — Build dataset.pkl from Excel (Optional)
Skip this section if `dataset.pkl` already exists in `data/processed/`.

**Required input files** (place in `data/raw/`):
- `Internal_signals.xlsx` — Long-format CZE-IT signal data (sample_id, curve_name, x, y)
- `Internal_labels.xlsx` — 9-class labels (sample_id, final_comment)

**Optional input files:**
- `External_signals.xlsx` + `External_labels.xlsx` — External validation cohort
- `Internal_demographics.xlsx` + `External_demographics.xlsx` — Age & sex

In [ ]:
import sys
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/001_IT_ML_github')
sys.path.insert(0, str(BASE))

In [ ]:
PROCESSED = BASE / 'data' / 'processed'
RAW = BASE / 'data' / 'raw'

In [ ]:
if not (PROCESSED / 'dataset.pkl').exists():
    print('dataset.pkl not found — building from Excel files...')
    from src.data_loader import build_dataset

    build_dataset(
        dev_signals=str(RAW / 'Internal_signals.xlsx'),
        dev_labels=str(RAW / 'Internal_labels.xlsx'),
        ext_signals=str(RAW / 'External_signals.xlsx') if (RAW / 'External_signals.xlsx').exists() else None,
        ext_labels=str(RAW / 'External_labels.xlsx') if (RAW / 'External_labels.xlsx').exists() else None,
        dev_demographics=str(RAW / 'Internal_demographics.xlsx') if (RAW / 'Internal_demographics.xlsx').exists() else None,
        ext_demographics=str(RAW / 'External_demographics.xlsx') if (RAW / 'External_demographics.xlsx').exists() else None,
        output_path=str(PROCESSED / 'dataset.pkl'),
        demographics_path=str(PROCESSED / 'demographics.xlsx'),
    )
else:
    print(f'dataset.pkl already exists ({(PROCESSED / "dataset.pkl").stat().st_size/1e6:.1f} MB) — skipping.')
    # Optional: validate existing pkl
    # from src.data_loader import validate_dataset
    # validate_dataset(str(PROCESSED / 'dataset.pkl'))

══════════════════════════════════════════════════════════════
0. SETUP & CONFIGURATION
══════════════════════════════════════════════════════════════

## 0. Setup & Configuration

In [ ]:
# --- Install dependencies (Colab) ---
%%capture
!pip install pytorch-tabnet sktime optuna shap mapie

In [ ]:
import numpy as np
import pandas as pd
import pickle, json, os, sys, time, warnings
from pathlib import Path
from collections import Counter

In [ ]:
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 150
import seaborn as sns; sns.set_style('whitegrid')

In [ ]:
import xgboost as xgb
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import StratifiedKFold, learning_curve, PredefinedSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve,
    cohen_kappa_score, classification_report, brier_score_loss
)

In [ ]:
warnings.filterwarnings('ignore')

### 0.1 Imports

BASE and sys.path already set in Data Preparation section above
If running from here directly, uncomment:
from google.colab import drive; drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/001_IT_ML_github')
sys.path.insert(0, str(BASE))

In [ ]:
from src.constants import (
    REGIONS, CHANNELS, CLASS9_NAMES, L2_CLASSES, L3_CLASSES,
    L1_METRICS, L2_METRICS, SEED, N_FOLDS, CONFIG_PRESETS,
)
from src.features import extract_all_features
from src.evaluation import (
    eval_binary, eval_multiclass, run_cv, print_cv_summary,
    find_optimal_thresholds, compute_cascade_metrics, bootstrap_ci,
    attribute_errors,
)
from src.cascade import (
    assemble_cascade_oof, run_external_cascade, get_fold_models,
)
from src.calibration import (
    compute_ece, compute_mce, scaled_brier_score,
    calibrate_oof_isotonic, calibration_report,
)
from src.confidence import (
    compute_cascade_confidence, build_cascade_proba_9class,
    conformal_prediction, validate_compound_calibration,
    validate_level_independence,
)
from src.cds import cds_predict
from src.explainability import aggregate_shap_by_region
from src.utils import Timer, save_pickle, load_pickle

In [ ]:
# Project paths
PROCESSED = BASE / 'data' / 'processed'
MODELS    = BASE / 'models'
RESULTS   = BASE / 'results'
for p in [PROCESSED, MODELS, RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

In [ ]:
print(f'BASE: {BASE}')

### 0.2 Configuration

In [ ]:
# ============================================================
# >>> SINGLE CONTROL POINT <<<
# ============================================================
QUALITY = 'best'          # 'fast' | 'best'
# ============================================================

In [ ]:
CFG = CONFIG_PRESETS[QUALITY]
np.random.seed(SEED)

In [ ]:
print(f'Quality: {QUALITY}')
print(f'  XGBoost: n_est={CFG["xgboost"]["n_estimators"]}, '
      f'depth={CFG["xgboost"]["max_depth"]}')
print(f'  CNN: epochs={CFG["cnn"]["epochs"]}, '
      f'filters={CFG["cnn"]["filters"]}')
print(f'  Optuna: L1={CFG["optuna"]["n_trials_l1"]} trials')

### 0.3 Load Data

In [ ]:
with open(PROCESSED / 'dataset.pkl', 'rb') as f:
    D = pickle.load(f)

In [ ]:
X_3d       = D['X_3d']           # (2219, 6, 300)
y_binary   = D['y_binary_enc']   # (2219,) — 0=Negative, 1=Positive
y_class9   = D['y_class9']       # (2219,) — string labels
N = len(y_binary)

In [ ]:
# Encode 9-class labels (alphabetical order = CLASS9_NAMES order)
le_9class = LabelEncoder()
le_9class.fit(y_class9)
y_9enc = le_9class.transform(y_class9)

In [ ]:
print(f'Samples: {N}')
print(f'  Negative: {(y_binary == 0).sum()} | Positive: {(y_binary == 1).sum()}')

### 0.4 Fold Assignments

In [ ]:
fold_path = RESULTS / 'fold_assignments.pkl'
if fold_path.exists():
    fold_indices = load_pickle(fold_path)['fold_indices']
    print('Loaded existing fold assignments')
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_indices = np.zeros(N, dtype=np.int8)
    for i, (_, tei) in enumerate(skf.split(X_3d, y_binary)):
        fold_indices[tei] = i
    save_pickle({'fold_indices': fold_indices, 'n_folds': N_FOLDS,
                 'seed': SEED}, fold_path)

In [ ]:
print(f'Folds: {N_FOLDS} | Distribution: {Counter(fold_indices)}')

### 0.5 Load Demographics

In [ ]:
demog_path = PROCESSED / 'demographics.xlsx'
if demog_path.exists():
    demog_all = pd.read_excel(demog_path)
    demog_train = demog_all[demog_all['cohort'] == 'Development'].reset_index(drop=True)
    demog_ext = demog_all[demog_all['cohort'] == 'External'].reset_index(drop=True)
    has_demog = True
    print(f'Demographics: train={len(demog_train)}, external={len(demog_ext)}')
    print(f'  Train — Age: {demog_train["age"].mean():.1f} ± {demog_train["age"].std():.1f}, '
          f'Female: {(demog_train["sex"] == "Female").sum()} ({100*(demog_train["sex"] == "Female").mean():.1f}%)')
else:
    has_demog = False
    demog_train = None
    demog_ext = None
    print('Demographics not found — fairness analysis will be skipped')

══════════════════════════════════════════════════════════════
1. PEAK FEATURE EXTRACTION
══════════════════════════════════════════════════════════════

## 1. Peak Feature Extraction
- β₁ (133–171): transferrin region
- β₂ (171–194): complement region
- β₂–γ transition (194–211)
- γ (211–263): immunoglobulin region
- M-protein zone (133–263)

In [ ]:
peak_path = PROCESSED / 'peak_features.pkl'
if peak_path.exists():
    PF = load_pickle(peak_path)
    X_peak = PF['X_peak']
    peak_cols = PF['columns']
    print(f'Loaded peak features: {X_peak.shape}')
else:
    print('Extracting peak features...')
    with Timer() as t:
        df_peak = extract_all_features(X_3d, CHANNELS)
    df_peak = df_peak.replace([np.inf, -np.inf], 0).fillna(0)
    X_peak = df_peak.values.astype(np.float32)
    peak_cols = list(df_peak.columns)
    save_pickle({'X_peak': X_peak, 'columns': peak_cols}, peak_path)
    print(f'Extracted {len(peak_cols)} features in {t.elapsed:.1f}s')

In [ ]:
# Top features correlated with positive class
corr = pd.DataFrame(X_peak, columns=peak_cols).corrwith(
    pd.Series(y_binary)).abs().sort_values(ascending=False)
print('Top 10 features correlated with POSITIVE:')
for name, val in corr.head(10).items():
    print(f'  {name:<45s} r={val:.4f}')

══════════════════════════════════════════════════════════════
2. FLAT BASELINES (9-CLASS)
══════════════════════════════════════════════════════════════

## 2. Flat Baselines — 9-Class Direct Classification
Flat LR and flat XGBoost as baselines for cascade comparison.
Uses the **same fold assignments** as the cascade for fair comparison.

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
print('=== Flat Baseline Models (same folds) ===')

In [ ]:
# --- Flat Logistic Regression ---
print('  Training flat Logistic Regression...')
oof_pred_lr = np.zeros(N, dtype=np.int64)
oof_proba_lr = np.zeros((N, 9), dtype=np.float32)

In [ ]:
with Timer() as t_lr:
    for fn in range(N_FOLDS):
        te = fold_indices == fn
        tri, tei = np.where(~te)[0], np.where(te)[0]
        lr = LogisticRegression(max_iter=2000, multi_class='multinomial',
                                solver='lbfgs', random_state=SEED, C=1.0)
        lr.fit(X_peak[tri], y_9enc[tri])
        oof_pred_lr[tei] = lr.predict(X_peak[tei])
        oof_proba_lr[tei] = lr.predict_proba(X_peak[tei])

In [ ]:
lr_acc = accuracy_score(y_9enc, oof_pred_lr)
lr_f1m = f1_score(y_9enc, oof_pred_lr, average='macro', zero_division=0)
lr_f1w = f1_score(y_9enc, oof_pred_lr, average='weighted', zero_division=0)
lr_kappa = cohen_kappa_score(y_9enc, oof_pred_lr)
print(f'  LR:  Acc={lr_acc:.4f}  Macro-F1={lr_f1m:.4f}  '
      f'WF1={lr_f1w:.4f}  κ={lr_kappa:.4f}  ({t_lr.elapsed:.1f}s)')

In [ ]:
# --- Flat XGBoost ---
print('  Training flat XGBoost...')
oof_pred_xgb_flat = np.zeros(N, dtype=np.int64)
oof_proba_xgb_flat = np.zeros((N, 9), dtype=np.float32)

In [ ]:
with Timer() as t_xgb:
    for fn in range(N_FOLDS):
        te = fold_indices == fn
        tri, tei = np.where(~te)[0], np.where(te)[0]
        m = xgb.XGBClassifier(
            n_estimators=500, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.5,
            eval_metric='mlogloss', use_label_encoder=False,
            random_state=SEED, early_stopping_rounds=50)
        m.fit(X_peak[tri], y_9enc[tri],
              eval_set=[(X_peak[tei], y_9enc[tei])], verbose=False)
        oof_pred_xgb_flat[tei] = m.predict(X_peak[tei])
        oof_proba_xgb_flat[tei] = m.predict_proba(X_peak[tei])

In [ ]:
xgb_acc = accuracy_score(y_9enc, oof_pred_xgb_flat)
xgb_f1m = f1_score(y_9enc, oof_pred_xgb_flat, average='macro', zero_division=0)
xgb_f1w = f1_score(y_9enc, oof_pred_xgb_flat, average='weighted', zero_division=0)
xgb_kappa = cohen_kappa_score(y_9enc, oof_pred_xgb_flat)
print(f'  XGB: Acc={xgb_acc:.4f}  Macro-F1={xgb_f1m:.4f}  '
      f'WF1={xgb_f1w:.4f}  κ={xgb_kappa:.4f}  ({t_xgb.elapsed:.1f}s)')

In [ ]:
# Save flat baselines
flat_lr_labels = le_9class.inverse_transform(oof_pred_lr)
flat_xgb_labels = le_9class.inverse_transform(oof_pred_xgb_flat)

In [ ]:
save_pickle({
    'flat_lr':  {'pred': flat_lr_labels, 'pred_enc': oof_pred_lr,
                 'proba': oof_proba_lr},
    'flat_xgb': {'pred': flat_xgb_labels, 'pred_enc': oof_pred_xgb_flat,
                 'proba': oof_proba_xgb_flat},
    'label_encoder_classes': le_9class.classes_.tolist(),
    'metrics': {
        'LR':  {'accuracy': lr_acc, 'macro_f1': lr_f1m,
                 'weighted_f1': lr_f1w, 'kappa': lr_kappa},
        'XGB': {'accuracy': xgb_acc, 'macro_f1': xgb_f1m,
                 'weighted_f1': xgb_f1w, 'kappa': xgb_kappa},
    }
}, RESULTS / 'L4_baseline_comparison.pkl')

══════════════════════════════════════════════════════════════
3. LEVEL 1 — BINARY M-PROTEIN DETECTION
══════════════════════════════════════════════════════════════

## 3. Level 1 — Binary M-Protein Detection (Negative vs Positive)
Seven model architectures compared; XGBoost-Peak-Optuna selected.

In [ ]:
def get_folds_l1():
    """Yield (train_idx, test_idx) for each fold."""
    for i in range(N_FOLDS):
        te = fold_indices == i
        yield np.where(~te)[0], np.where(te)[0]

### 3a. XGBoost — Raw Signal Features (1800-d)

In [ ]:
p = CFG['xgboost']
X_raw = D['X']  # (2219, 1800) flattened 6×300
print(f'=== L1: XGBoost-Raw [{QUALITY}] ===')

In [ ]:
def train_xgb_raw(Xtr, ytr, Xte, yte, fn):
    return xgb.XGBClassifier(
        n_estimators=p['n_estimators'], max_depth=p['max_depth'],
        learning_rate=p['learning_rate'], subsample=p['subsample'],
        colsample_bytree=p['colsample_bytree'], reg_alpha=p['reg_alpha'],
        reg_lambda=p['reg_lambda'], min_child_weight=3,
        early_stopping_rounds=p['early_stopping_rounds'],
        eval_metric='logloss', use_label_encoder=False, random_state=SEED
    ).fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)

In [ ]:
with Timer() as t:
    oof_p_xgb_raw, oof_c_xgb_raw, cv_xgb_raw, _ = run_cv(
        train_xgb_raw, X_raw, fold_indices, y_binary,
        L1_METRICS, eval_binary, label='XGB-Raw')
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_xgb_raw)

### 3b. XGBoost — Peak Features (399-d)

In [ ]:
print(f'=== L1: XGBoost-Peak [{QUALITY}] ===')

In [ ]:
def train_xgb_peak(Xtr, ytr, Xte, yte, fn):
    return xgb.XGBClassifier(
        n_estimators=p['n_estimators'], max_depth=p['max_depth'],
        learning_rate=p['learning_rate'], subsample=p['subsample'],
        colsample_bytree=p['colsample_bytree'], reg_alpha=p['reg_alpha'],
        reg_lambda=p['reg_lambda'], min_child_weight=3,
        early_stopping_rounds=p['early_stopping_rounds'],
        eval_metric='logloss', use_label_encoder=False, random_state=SEED
    ).fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)

In [ ]:
with Timer() as t:
    oof_p_xgb_pk, oof_c_xgb_pk, cv_xgb_pk, xpk_models = run_cv(
        train_xgb_peak, X_peak, fold_indices, y_binary,
        L1_METRICS, eval_binary, label='XGB-Peak')
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_xgb_pk)

### 3c. 1D-CNN

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.random.set_seed(SEED)

In [ ]:
p_cnn = CFG['cnn']
print(f'=== L1: 1D-CNN [{QUALITY}] ===')

In [ ]:
def build_cnn_l1():
    inp = layers.Input(shape=(6, 300))
    x = layers.Permute((2, 1))(inp)
    for fl, ks in zip(p_cnn['filters'], p_cnn['kernel_sizes']):
        x = layers.Conv1D(fl, kernel_size=ks, padding='same')(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.MaxPooling1D(pool_size=3)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(p_cnn['dropout_1'])(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(p_cnn['dropout_2'])(x)
    x = layers.Dense(1, activation='sigmoid')(x)
    m = keras.Model(inp, x)
    m.compile(optimizer=keras.optimizers.Adam(p_cnn['lr']),
              loss='binary_crossentropy', metrics=['accuracy'])
    return m

In [ ]:
oof_p_cnn = np.zeros(N, dtype=np.float32)
oof_c_cnn = np.zeros(N, dtype=np.int8)
cv_cnn = {m: [] for m in L1_METRICS}

In [ ]:
with Timer() as t:
    for fn, (tri, tei) in enumerate(get_folds_l1()):
        m = build_cnn_l1()
        m.fit(X_3d[tri], y_binary[tri],
              validation_data=(X_3d[tei], y_binary[tei]),
              epochs=p_cnn['epochs'], batch_size=p_cnn['batch_size'],
              callbacks=[keras.callbacks.EarlyStopping(
                  patience=p_cnn['patience'], restore_best_weights=True)],
              verbose=0)
        prob = m.predict(X_3d[tei], verbose=0).ravel()
        pred = (prob >= 0.5).astype(np.int8)
        oof_p_cnn[tei] = prob; oof_c_cnn[tei] = pred
        met = eval_binary(y_binary[tei], pred, prob)
        for k in L1_METRICS:
            if k in met: cv_cnn[k].append(met[k])
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_cnn)

### 3d. MiniRocket

In [ ]:
from sktime.transformations.panel.rocket import MiniRocket
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.special import expit

In [ ]:
p_rkt = CFG['rocket']
print(f'=== L1: MiniRocket [{QUALITY}] ===')

In [ ]:
oof_p_rkt = np.zeros(N, dtype=np.float32)
oof_c_rkt = np.zeros(N, dtype=np.int8)
cv_rkt = {m: [] for m in L1_METRICS}

In [ ]:
with Timer() as t:
    for fn, (tri, tei) in enumerate(get_folds_l1()):
        r = MiniRocket(num_kernels=p_rkt['num_kernels'], random_state=SEED)
        r.fit(X_3d[tri])
        Xr_tr = r.transform(X_3d[tri])
        Xr_te = r.transform(X_3d[tei])
        clf = make_pipeline(StandardScaler(),
                            RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)))
        clf.fit(Xr_tr, y_binary[tri])
        pred = clf.predict(Xr_te)
        prob = expit(clf.decision_function(Xr_te))
        oof_p_rkt[tei] = prob; oof_c_rkt[tei] = pred
        met = eval_binary(y_binary[tei], pred, prob)
        for k in L1_METRICS:
            if k in met: cv_rkt[k].append(met[k])
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_rkt)

### 3e. MiniRocket + Peak (Hybrid)

In [ ]:
print(f'=== L1: MiniRocket+Peak [{QUALITY}] ===')
oof_p_rkpk = np.zeros(N, dtype=np.float32)
oof_c_rkpk = np.zeros(N, dtype=np.int8)
cv_rkpk = {m: [] for m in L1_METRICS}

In [ ]:
with Timer() as t:
    for fn, (tri, tei) in enumerate(get_folds_l1()):
        r = MiniRocket(num_kernels=p_rkt['num_kernels'], random_state=SEED)
        r.fit(X_3d[tri])
        Xr_tr = np.hstack([r.transform(X_3d[tri]), X_peak[tri]])
        Xr_te = np.hstack([r.transform(X_3d[tei]), X_peak[tei]])
        clf = make_pipeline(StandardScaler(),
                            RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)))
        clf.fit(Xr_tr, y_binary[tri])
        pred = clf.predict(Xr_te)
        prob = expit(clf.decision_function(Xr_te))
        oof_p_rkpk[tei] = prob; oof_c_rkpk[tei] = pred
        met = eval_binary(y_binary[tei], pred, prob)
        for k in L1_METRICS:
            if k in met: cv_rkpk[k].append(met[k])
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_rkpk)

### 3f. TabNet

In [ ]:
from pytorch_tabnet.tab_model import TabNetClassifier
import torch; torch.manual_seed(SEED)

In [ ]:
p_tab = CFG['tabnet']
print(f'=== L1: TabNet [{QUALITY}] ===')

In [ ]:
oof_p_tab = np.zeros(N, dtype=np.float32)
oof_c_tab = np.zeros(N, dtype=np.int8)
cv_tab = {m: [] for m in L1_METRICS}

In [ ]:
with Timer() as t:
    for fn, (tri, tei) in enumerate(get_folds_l1()):
        clf = TabNetClassifier(
            n_d=p_tab['n_d'], n_a=p_tab['n_a'], n_steps=p_tab['n_steps'],
            gamma=p_tab['gamma'], lambda_sparse=p_tab['lambda_sparse'],
            mask_type=p_tab['mask_type'], seed=SEED, verbose=0)
        clf.fit(X_peak[tri], y_binary[tri],
                eval_set=[(X_peak[tei], y_binary[tei])],
                max_epochs=p_tab['max_epochs'], patience=p_tab['patience'],
                batch_size=p_tab['batch_size'],
                virtual_batch_size=p_tab['virtual_batch_size'])
        prob = clf.predict_proba(X_peak[tei])[:, 1]
        pred = (prob >= 0.5).astype(np.int8)
        oof_p_tab[tei] = prob; oof_c_tab[tei] = pred
        met = eval_binary(y_binary[tei], pred, prob)
        for k in L1_METRICS:
            if k in met: cv_tab[k].append(met[k])
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_tab)

### 3g. L1 Model Comparison

In [ ]:
all_cv_l1 = {
    'XGBoost-Raw': cv_xgb_raw, 'XGBoost-Peak': cv_xgb_pk,
    '1D-CNN': cv_cnn, 'MiniRocket': cv_rkt,
    'MiniRocket+Peak': cv_rkpk, 'TabNet': cv_tab,
}
all_proba_l1 = {
    'XGBoost-Raw': oof_p_xgb_raw, 'XGBoost-Peak': oof_p_xgb_pk,
    '1D-CNN': oof_p_cnn, 'MiniRocket': oof_p_rkt,
    'MiniRocket+Peak': oof_p_rkpk, 'TabNet': oof_p_tab,
}

In [ ]:
ranked = sorted(all_cv_l1.items(),
                key=lambda x: np.mean(x[1].get('auc_roc', [0])), reverse=True)
print(f'\n{"Model":<25s}', end='')
for m in L1_METRICS: print(f' {m:>14s}', end='')
print()
print('-' * 100)
for name, cv in ranked:
    print(f'{name:<25s}', end='')
    for m in L1_METRICS:
        vals = cv.get(m, [])
        if vals: print(f' {np.mean(vals):>7.4f}±{np.std(vals):.4f}', end='')
        else: print(f' {"N/A":>14s}', end='')
    print()

### 3h. L1 Threshold Optimization

In [ ]:
thresh_results = {}
for name, proba in all_proba_l1.items():
    thresh_results[name] = find_optimal_thresholds(y_binary, proba, name)

In [ ]:
print(f'\n{"Model":<25s} {"Youden Thr":>10s} {"Sens":>7s} {"Spec":>7s}')
print('-' * 55)
for name, tr in thresh_results.items():
    yo = tr['youden']
    print(f'{name:<25s} {yo["threshold"]:>10.4f} '
          f'{yo["sensitivity"]:>7.4f} {yo["specificity"]:>7.4f}')

### 3i. L1 Ensemble

In [ ]:
print('\n=== Ensemble Strategies ===')
top2 = [ranked[0][0], ranked[1][0]]
top2_p = [all_proba_l1[top2[0]], all_proba_l1[top2[1]]]

In [ ]:
ens_avg = (top2_p[0] + top2_p[1]) / 2
ens_max = np.maximum(top2_p[0], top2_p[1])
a0 = roc_auc_score(y_binary, top2_p[0])
a1 = roc_auc_score(y_binary, top2_p[1])
w0, w1 = a0 / (a0 + a1), a1 / (a0 + a1)
ens_wt = w0 * top2_p[0] + w1 * top2_p[1]

In [ ]:
for name, prob in [(f'{top2[0]}', top2_p[0]), (f'{top2[1]}', top2_p[1]),
                   ('Average', ens_avg), ('Max', ens_max), ('Weighted', ens_wt)]:
    print(f'  {name:<30s}  AUC = {roc_auc_score(y_binary, prob):.5f}')

### 3j. L1 XGBoost-Peak — Optuna Optimization

In [ ]:
N_TRIALS_L1 = CFG['optuna']['n_trials_l1']
print(f'\n=== XGBoost-Peak Optuna ({N_TRIALS_L1} trials) ===')

In [ ]:
def xgb_peak_l1_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 1500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    aucs = []
    for fn, (tri, tei) in enumerate(get_folds_l1()):
        m = xgb.XGBClassifier(**params, eval_metric='logloss',
                              use_label_encoder=False, random_state=SEED,
                              early_stopping_rounds=50)
        m.fit(X_peak[tri], y_binary[tri],
              eval_set=[(X_peak[tei], y_binary[tei])], verbose=False)
        aucs.append(roc_auc_score(y_binary[tei], m.predict_proba(X_peak[tei])[:, 1]))
    return np.mean(aucs)

In [ ]:
with Timer() as t_opt:
    study_l1 = optuna.create_study(direction='maximize', study_name='L1_XGB_Peak',
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
    study_l1.optimize(xgb_peak_l1_objective, n_trials=N_TRIALS_L1,
                      show_progress_bar=True)

In [ ]:
print(f'Best AUC: {study_l1.best_value:.5f} ({t_opt.elapsed:.0f}s)')
print(f'Best params: {study_l1.best_params}')

In [ ]:
# Retrain with best params
bp_l1 = study_l1.best_params
oof_p_xpk_opt = np.zeros(N, dtype=np.float32)
oof_c_xpk_opt = np.zeros(N, dtype=np.int8)
cv_xpk_opt = {m: [] for m in L1_METRICS}
l1_opt_models = []

In [ ]:
with Timer() as t_re:
    for fn, (tri, tei) in enumerate(get_folds_l1()):
        m = xgb.XGBClassifier(**bp_l1, eval_metric='logloss',
                              use_label_encoder=False, random_state=SEED,
                              early_stopping_rounds=50)
        m.fit(X_peak[tri], y_binary[tri],
              eval_set=[(X_peak[tei], y_binary[tei])], verbose=False)
        prob = m.predict_proba(X_peak[tei])[:, 1]
        pred = (prob >= 0.5).astype(np.int8)
        oof_p_xpk_opt[tei] = prob; oof_c_xpk_opt[tei] = pred
        met = eval_binary(y_binary[tei], pred, prob)
        for k in L1_METRICS:
            if k in met: cv_xpk_opt[k].append(met[k])
        l1_opt_models.append(m)

In [ ]:
print(f'  ({t_re.elapsed:.1f}s)'); print_cv_summary(cv_xpk_opt)

In [ ]:
# Add to comparison
all_cv_l1['XGBoost-Peak-Optuna'] = cv_xpk_opt
all_proba_l1['XGBoost-Peak-Optuna'] = oof_p_xpk_opt
thresh_results['XGBoost-Peak-Optuna'] = find_optimal_thresholds(
    y_binary, oof_p_xpk_opt, 'XGB-Peak-Optuna')

### 3k. Save L1 Results

In [ ]:
L1_results = {
    'oof_proba': all_proba_l1,
    'cv_metrics': all_cv_l1,
    'y_true': y_binary,
    'threshold_analysis': thresh_results,
    'optuna_best_params': bp_l1,
}
save_pickle(L1_results, RESULTS / 'L1_oof_predictions.pkl')
save_pickle(l1_opt_models, MODELS / 'L1_xgb_peak_optuna_models.pkl')
print('L1 results saved ✓')

══════════════════════════════════════════════════════════════
4. LEVEL 2 — HEAVY CHAIN CLASSIFICATION (IgG / IgA / IgM / Free)
══════════════════════════════════════════════════════════════

## 4. Level 2 — Heavy Chain Classification
4-class classifier on L1-positive samples only.

In [ ]:
# Positive subset
pos_mask = (y_binary == 1)
pos_idx = np.where(pos_mask)[0]
N_pos = len(pos_idx)

In [ ]:
y_heavy_pos = D['y_heavy'][pos_idx]
y_light_pos = D['y_light'][pos_idx]

In [ ]:
X_l2       = X_peak[pos_idx]
X_3d_l2    = X_3d[pos_idx]
fold_idx_pos = fold_indices[pos_idx]

In [ ]:
le_heavy = LabelEncoder()
le_heavy.classes_ = np.array(L2_CLASSES)
y_l2 = le_heavy.transform(y_heavy_pos)
N_CLS_L2 = len(L2_CLASSES)

In [ ]:
cw = compute_class_weight('balanced', classes=np.arange(N_CLS_L2), y=y_l2)

In [ ]:
print(f'Positive samples: {N_pos}')
for i, cls in enumerate(L2_CLASSES):
    print(f'  {cls}: {(y_l2 == i).sum()}')

In [ ]:
def get_folds_l2():
    for i in range(N_FOLDS):
        te = fold_idx_pos == i
        yield np.where(~te)[0], np.where(te)[0]

In [ ]:
L2_MET = L2_METRICS + [f'recall_{c}' for c in L2_CLASSES]

### 4a. L2 XGBoost-Peak

In [ ]:
p = CFG['xgboost']
print(f'=== L2: XGBoost-Peak [{QUALITY}] ===')

In [ ]:
oof_prob_l2_xpk = np.zeros((N_pos, N_CLS_L2), dtype=np.float32)
oof_pred_l2_xpk = np.zeros(N_pos, dtype=np.int8)
cv_l2_xpk = {m: [] for m in L2_MET}
l2_xpk_models = []

In [ ]:
with Timer() as t:
    for fn, (tri, tei) in enumerate(get_folds_l2()):
        sw = np.array([cw[c] for c in y_l2[tri]])
        m = xgb.XGBClassifier(
            objective='multi:softprob', num_class=N_CLS_L2,
            n_estimators=p['n_estimators'], max_depth=p['max_depth'],
            learning_rate=p['learning_rate'], subsample=p['subsample'],
            colsample_bytree=p['colsample_bytree'], reg_alpha=p['reg_alpha'],
            reg_lambda=p['reg_lambda'], min_child_weight=3,
            early_stopping_rounds=p['early_stopping_rounds'],
            eval_metric='mlogloss', use_label_encoder=False, random_state=SEED
        ).fit(X_l2[tri], y_l2[tri], eval_set=[(X_l2[tei], y_l2[tei])],
              sample_weight=sw, verbose=False)
        prob = m.predict_proba(X_l2[tei])
        pred = np.argmax(prob, axis=1)
        oof_prob_l2_xpk[tei] = prob; oof_pred_l2_xpk[tei] = pred
        met = eval_multiclass(y_l2[tei], pred, prob, N_CLS_L2)
        for k in L2_MET:
            if k in met: cv_l2_xpk[k].append(met[k])
        l2_xpk_models.append(m)
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_l2_xpk, L2_METRICS)

### 4b. L2 1D-CNN

In [ ]:
tf.random.set_seed(SEED)
p_cnn = CFG['cnn']
print(f'=== L2: 1D-CNN [{QUALITY}] ===')

In [ ]:
def build_cnn_l2():
    inp = layers.Input(shape=(6, 300))
    x = layers.Permute((2, 1))(inp)
    for fl, ks in zip(p_cnn['filters'], p_cnn['kernel_sizes']):
        x = layers.Conv1D(fl, kernel_size=ks, padding='same')(x)
        x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.MaxPooling1D(pool_size=3)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(p_cnn['dropout_1'])(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(p_cnn['dropout_2'])(x)
    x = layers.Dense(N_CLS_L2, activation='softmax')(x)
    m = keras.Model(inp, x)
    m.compile(optimizer=keras.optimizers.Adam(p_cnn['lr']),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

In [ ]:
oof_prob_l2_cnn = np.zeros((N_pos, N_CLS_L2), dtype=np.float32)
oof_pred_l2_cnn = np.zeros(N_pos, dtype=np.int8)
cv_l2_cnn = {m: [] for m in L2_MET}

In [ ]:
with Timer() as t:
    for fn, (tri, tei) in enumerate(get_folds_l2()):
        m = build_cnn_l2()
        m.fit(X_3d_l2[tri], y_l2[tri],
              validation_data=(X_3d_l2[tei], y_l2[tei]),
              epochs=p_cnn['epochs'], batch_size=p_cnn['batch_size'],
              callbacks=[keras.callbacks.EarlyStopping(
                  patience=p_cnn['patience'], restore_best_weights=True)],
              verbose=0)
        prob = m.predict(X_3d_l2[tei], verbose=0)
        pred = np.argmax(prob, axis=1)
        oof_prob_l2_cnn[tei] = prob; oof_pred_l2_cnn[tei] = pred
        met = eval_multiclass(y_l2[tei], pred, prob, N_CLS_L2)
        for k in L2_MET:
            if k in met: cv_l2_cnn[k].append(met[k])
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_l2_cnn, L2_METRICS)

### 4c. L2 MiniRocket

In [ ]:
print(f'=== L2: MiniRocket [{QUALITY}] ===')
oof_prob_l2_rkt = np.zeros((N_pos, N_CLS_L2), dtype=np.float32)
oof_pred_l2_rkt = np.zeros(N_pos, dtype=np.int8)
cv_l2_rkt = {m: [] for m in L2_MET}

In [ ]:
with Timer() as t:
    for fn, (tri, tei) in enumerate(get_folds_l2()):
        r = MiniRocket(num_kernels=p_rkt['num_kernels'], random_state=SEED)
        r.fit(X_3d_l2[tri])
        Xr_tr = r.transform(X_3d_l2[tri])
        Xr_te = r.transform(X_3d_l2[tei])
        clf = make_pipeline(StandardScaler(),
                            RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)))
        clf.fit(Xr_tr, y_l2[tri])
        pred = clf.predict(Xr_te)
        # Ridge has no predict_proba — use softmax on decision function
        dec = clf.decision_function(Xr_te)
        prob = np.exp(dec) / np.exp(dec).sum(axis=1, keepdims=True)
        oof_prob_l2_rkt[tei] = prob; oof_pred_l2_rkt[tei] = pred
        met = eval_multiclass(y_l2[tei], pred, prob, N_CLS_L2)
        for k in L2_MET:
            if k in met: cv_l2_rkt[k].append(met[k])
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_l2_rkt, L2_METRICS)

### 4d. L2 XGBoost-Peak — Optuna

In [ ]:
N_TRIALS_L2 = CFG['optuna']['n_trials_l2']
print(f'=== L2: XGBoost-Peak Optuna ({N_TRIALS_L2} trials) ===')

In [ ]:
def xgb_l2_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 1500),
        'max_depth':        trial.suggest_int('max_depth', 3, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    f1s = []
    for fn, (tri, tei) in enumerate(get_folds_l2()):
        sw = np.array([cw[c] for c in y_l2[tri]])
        m = xgb.XGBClassifier(**params, objective='multi:softprob',
                              num_class=N_CLS_L2, eval_metric='mlogloss',
                              use_label_encoder=False, random_state=SEED,
                              early_stopping_rounds=50)
        m.fit(X_l2[tri], y_l2[tri], eval_set=[(X_l2[tei], y_l2[tei])],
              sample_weight=sw, verbose=False)
        f1s.append(f1_score(y_l2[tei], m.predict(X_l2[tei]), average='macro'))
    return np.mean(f1s)

In [ ]:
with Timer() as t_opt:
    study_l2 = optuna.create_study(direction='maximize', study_name='L2_XGB_Peak',
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
    study_l2.optimize(xgb_l2_objective, n_trials=N_TRIALS_L2,
                      show_progress_bar=True)
print(f'Best macro_f1: {study_l2.best_value:.5f} ({t_opt.elapsed:.0f}s)')

In [ ]:
bp_l2 = study_l2.best_params
oof_prob_l2_opt = np.zeros((N_pos, N_CLS_L2), dtype=np.float32)
oof_pred_l2_opt = np.zeros(N_pos, dtype=np.int8)
cv_l2_opt = {m: [] for m in L2_MET}
l2_opt_models = []

In [ ]:
with Timer() as t_re:
    for fn, (tri, tei) in enumerate(get_folds_l2()):
        sw = np.array([cw[c] for c in y_l2[tri]])
        m = xgb.XGBClassifier(**bp_l2, objective='multi:softprob',
                              num_class=N_CLS_L2, eval_metric='mlogloss',
                              use_label_encoder=False, random_state=SEED,
                              early_stopping_rounds=50)
        m.fit(X_l2[tri], y_l2[tri], eval_set=[(X_l2[tei], y_l2[tei])],
              sample_weight=sw, verbose=False)
        prob = m.predict_proba(X_l2[tei])
        pred = np.argmax(prob, axis=1)
        oof_prob_l2_opt[tei] = prob; oof_pred_l2_opt[tei] = pred
        met = eval_multiclass(y_l2[tei], pred, prob, N_CLS_L2)
        for k in L2_MET:
            if k in met: cv_l2_opt[k].append(met[k])
        l2_opt_models.append(m)
print(f'  ({t_re.elapsed:.1f}s)'); print_cv_summary(cv_l2_opt, L2_METRICS)

### 4e. L2 Comparison & Save

In [ ]:
all_cv_l2 = {'XGBoost-Peak': cv_l2_xpk, '1D-CNN': cv_l2_cnn,
             'MiniRocket': cv_l2_rkt, 'XGBoost-Peak-Optuna': cv_l2_opt}
all_prob_l2 = {'XGBoost-Peak': oof_prob_l2_xpk, '1D-CNN': oof_prob_l2_cnn,
               'MiniRocket': oof_prob_l2_rkt, 'XGBoost-Peak-Optuna': oof_prob_l2_opt}
all_pred_l2 = {'XGBoost-Peak': oof_pred_l2_xpk, '1D-CNN': oof_pred_l2_cnn,
               'MiniRocket': oof_pred_l2_rkt, 'XGBoost-Peak-Optuna': oof_pred_l2_opt}

In [ ]:
ranked_l2 = sorted(all_cv_l2.items(),
                   key=lambda x: np.mean(x[1]['macro_f1']), reverse=True)
print(f'\n{"Model":<25s}', end='')
for m in L2_METRICS: print(f' {m:>12s}', end='')
print()
for name, cv in ranked_l2:
    print(f'{name:<25s}', end='')
    for m in L2_METRICS: print(f' {np.mean(cv[m]):>12.4f}', end='')
    print()

In [ ]:
save_pickle({'oof_prob': all_prob_l2, 'oof_pred': all_pred_l2,
             'cv_metrics': all_cv_l2, 'y_true': y_l2,
             'classes': L2_CLASSES, 'pos_idx': pos_idx,
             'optuna_best_params': bp_l2},
            RESULTS / 'L2_oof_predictions.pkl')
save_pickle(l2_opt_models, MODELS / 'L2_xgb_peak_optuna_models.pkl')
print('L2 results saved ✓')

══════════════════════════════════════════════════════════════
5. LEVEL 3 — LIGHT CHAIN CLASSIFICATION (κ vs λ)
══════════════════════════════════════════════════════════════

## 5. Level 3 — Light Chain Classification (Kappa vs Lambda)
Binary classification on positive samples. Same data split as L2.

In [ ]:
le_light = LabelEncoder()
le_light.classes_ = np.array(L3_CLASSES)
y_l3 = le_light.transform(y_light_pos)
print(f'L3 target: KAPPA={(y_l3==0).sum()}, LAMBDA={(y_l3==1).sum()}')

In [ ]:
def get_folds_l3():
    for i in range(N_FOLDS):
        te = fold_idx_pos == i
        yield np.where(~te)[0], np.where(te)[0]

### 5a. L3 XGBoost-Peak

In [ ]:
p = CFG['xgboost']
print(f'=== L3: XGBoost-Peak [{QUALITY}] ===')

In [ ]:
oof_p_l3_xpk = np.zeros(N_pos, dtype=np.float32)
cv_l3_xpk = {m: [] for m in L1_METRICS}

In [ ]:
with Timer() as t:
    for fn, (tri, tei) in enumerate(get_folds_l3()):
        m = xgb.XGBClassifier(
            n_estimators=p['n_estimators'], max_depth=p['max_depth'],
            learning_rate=p['learning_rate'], subsample=p['subsample'],
            colsample_bytree=p['colsample_bytree'], reg_alpha=p['reg_alpha'],
            reg_lambda=p['reg_lambda'], min_child_weight=3,
            early_stopping_rounds=p['early_stopping_rounds'],
            eval_metric='logloss', use_label_encoder=False, random_state=SEED
        ).fit(X_l2[tri], y_l3[tri], eval_set=[(X_l2[tei], y_l3[tei])],
              verbose=False)
        prob = m.predict_proba(X_l2[tei])[:, 1]
        pred = (prob >= 0.5).astype(np.int8)
        oof_p_l3_xpk[tei] = prob
        met = eval_binary(y_l3[tei], pred, prob)
        for k in L1_METRICS:
            if k in met: cv_l3_xpk[k].append(met[k])
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_l3_xpk)

### 5b. L3 1D-CNN

In [ ]:
tf.random.set_seed(SEED)
print(f'=== L3: 1D-CNN [{QUALITY}] ===')
oof_p_l3_cnn = np.zeros(N_pos, dtype=np.float32)
cv_l3_cnn = {m: [] for m in L1_METRICS}

In [ ]:
with Timer() as t:
    for fn, (tri, tei) in enumerate(get_folds_l3()):
        m = build_cnn_l1()  # same architecture as L1 binary
        m.fit(X_3d_l2[tri], y_l3[tri],
              validation_data=(X_3d_l2[tei], y_l3[tei]),
              epochs=p_cnn['epochs'], batch_size=p_cnn['batch_size'],
              callbacks=[keras.callbacks.EarlyStopping(
                  patience=p_cnn['patience'], restore_best_weights=True)],
              verbose=0)
        prob = m.predict(X_3d_l2[tei], verbose=0).ravel()
        pred = (prob >= 0.5).astype(np.int8)
        oof_p_l3_cnn[tei] = prob
        met = eval_binary(y_l3[tei], pred, prob)
        for k in L1_METRICS:
            if k in met: cv_l3_cnn[k].append(met[k])
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_l3_cnn)

### 5c. L3 MiniRocket

In [ ]:
print(f'=== L3: MiniRocket [{QUALITY}] ===')
oof_p_l3_rkt = np.zeros(N_pos, dtype=np.float32)
cv_l3_rkt = {m: [] for m in L1_METRICS}

In [ ]:
with Timer() as t:
    for fn, (tri, tei) in enumerate(get_folds_l3()):
        r = MiniRocket(num_kernels=p_rkt['num_kernels'], random_state=SEED)
        r.fit(X_3d_l2[tri])
        clf = make_pipeline(StandardScaler(),
                            RidgeClassifierCV(alphas=np.logspace(-3, 3, 10)))
        clf.fit(r.transform(X_3d_l2[tri]), y_l3[tri])
        pred = clf.predict(r.transform(X_3d_l2[tei]))
        prob = expit(clf.decision_function(r.transform(X_3d_l2[tei])))
        oof_p_l3_rkt[tei] = prob
        met = eval_binary(y_l3[tei], pred, prob)
        for k in L1_METRICS:
            if k in met: cv_l3_rkt[k].append(met[k])
print(f'  ({t.elapsed:.1f}s)'); print_cv_summary(cv_l3_rkt)

### 5d. L3 XGBoost-Peak — Optuna

In [ ]:
N_TRIALS_L3 = CFG['optuna']['n_trials_l3']
print(f'=== L3: XGBoost-Peak Optuna ({N_TRIALS_L3} trials) ===')

In [ ]:
def xgb_l3_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 1500),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    aucs = []
    for fn, (tri, tei) in enumerate(get_folds_l3()):
        m = xgb.XGBClassifier(**params, eval_metric='logloss',
                              use_label_encoder=False, random_state=SEED,
                              early_stopping_rounds=50)
        m.fit(X_l2[tri], y_l3[tri], eval_set=[(X_l2[tei], y_l3[tei])],
              verbose=False)
        aucs.append(roc_auc_score(y_l3[tei], m.predict_proba(X_l2[tei])[:, 1]))
    return np.mean(aucs)

In [ ]:
with Timer() as t_opt:
    study_l3 = optuna.create_study(direction='maximize', study_name='L3_XGB_Peak',
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
    study_l3.optimize(xgb_l3_objective, n_trials=N_TRIALS_L3,
                      show_progress_bar=True)
print(f'Best AUC: {study_l3.best_value:.5f} ({t_opt.elapsed:.0f}s)')

In [ ]:
bp_l3 = study_l3.best_params
oof_p_l3_opt = np.zeros(N_pos, dtype=np.float32)
cv_l3_opt = {m: [] for m in L1_METRICS}
l3_opt_models = []

In [ ]:
with Timer() as t_re:
    for fn, (tri, tei) in enumerate(get_folds_l3()):
        m = xgb.XGBClassifier(**bp_l3, eval_metric='logloss',
                              use_label_encoder=False, random_state=SEED,
                              early_stopping_rounds=50)
        m.fit(X_l2[tri], y_l3[tri], eval_set=[(X_l2[tei], y_l3[tei])],
              verbose=False)
        prob = m.predict_proba(X_l2[tei])[:, 1]
        pred = (prob >= 0.5).astype(np.int8)
        oof_p_l3_opt[tei] = prob
        met = eval_binary(y_l3[tei], pred, prob)
        for k in L1_METRICS:
            if k in met: cv_l3_opt[k].append(met[k])
        l3_opt_models.append(m)
print(f'  ({t_re.elapsed:.1f}s)'); print_cv_summary(cv_l3_opt)

### 5e. L3 Comparison & Save

In [ ]:
all_cv_l3 = {'XGBoost-Peak': cv_l3_xpk, '1D-CNN': cv_l3_cnn,
             'MiniRocket': cv_l3_rkt, 'XGBoost-Peak-Optuna': cv_l3_opt}
all_prob_l3 = {'XGBoost-Peak': oof_p_l3_xpk, '1D-CNN': oof_p_l3_cnn,
               'MiniRocket': oof_p_l3_rkt, 'XGBoost-Peak-Optuna': oof_p_l3_opt}

In [ ]:
ranked_l3 = sorted(all_cv_l3.items(),
                   key=lambda x: np.mean(x[1].get('auc_roc', [0])), reverse=True)
print(f'\n{"Model":<25s}', end='')
for m in L1_METRICS: print(f' {m:>14s}', end='')
print()
for name, cv in ranked_l3:
    print(f'{name:<25s}', end='')
    for m in L1_METRICS:
        vals = cv.get(m, [])
        if vals: print(f' {np.mean(vals):>7.4f}±{np.std(vals):.4f}', end='')
    print()

In [ ]:
save_pickle({
    'oof_proba': all_prob_l3, 'oof_pred': {k: (v >= 0.5).astype(int) for k, v in all_prob_l3.items()},
    'cv_metrics': all_cv_l3, 'y_true': y_l3,
    'classes': L3_CLASSES, 'pos_idx': pos_idx,
    'optuna_best_params': bp_l3,
}, RESULTS / 'L3_oof_predictions.pkl')
save_pickle(l3_opt_models, MODELS / 'L3_xgb_peak_optuna_models.pkl')
print('L3 results saved ✓')

══════════════════════════════════════════════════════════════
6. CASCADE ASSEMBLY & 9-CLASS EVALUATION
══════════════════════════════════════════════════════════════

## 6. Cascade Assembly — L1 → L2 → L3 → 9-Class

In [ ]:
# Selected models: XGBoost-Peak-Optuna at all levels
L1_MODEL_KEY = 'XGBoost-Peak-Optuna'
L1_THRESHOLD = thresh_results[L1_MODEL_KEY]['youden']['threshold']
print(f'L1 model: {L1_MODEL_KEY}, threshold (Youden): {L1_THRESHOLD:.4f}')

In [ ]:
l1_proba = all_proba_l1[L1_MODEL_KEY]
l2_pred  = oof_pred_l2_opt
l3_pred  = (oof_p_l3_opt >= 0.5).astype(int)

In [ ]:
l2_models_loaded = l2_opt_models
l3_models_loaded = l3_opt_models

In [ ]:
best_pred, fp_idx, cascade_info = assemble_cascade_oof(
    l1_proba, l2_pred, l3_pred, L1_THRESHOLD,
    pos_idx, y_binary, l2_models_loaded, l3_models_loaded, X_peak)

In [ ]:
best_metrics = compute_cascade_metrics(y_class9, best_pred)

In [ ]:
print(f'\n=== CASCADE RESULTS ===')
print(f'  Accuracy:    {best_metrics["accuracy"]:.4f}')
print(f'  Macro F1:    {best_metrics["macro_f1"]:.4f}')
print(f'  Weighted F1: {best_metrics["weighted_f1"]:.4f}')
print(f'  Cohen κ:     {best_metrics["kappa"]:.4f}')
print(f'  FP samples:  {cascade_info["fp_count"]}')

In [ ]:
# Comparison with flat baselines
print(f'\n=== CASCADE vs FLAT ===')
print(f'  {"Model":<25s} {"Accuracy":>10s} {"Macro F1":>10s} {"Weighted F1":>12s} {"κ":>8s}')
print(f'  {"-"*70}')
print(f'  {"Flat LR":<25s} {lr_acc:>10.4f} {lr_f1m:>10.4f} {lr_f1w:>12.4f} {lr_kappa:>8.4f}')
print(f'  {"Flat XGBoost":<25s} {xgb_acc:>10.4f} {xgb_f1m:>10.4f} {xgb_f1w:>12.4f} {xgb_kappa:>8.4f}')
print(f'  {"CASCADE":<25s} {best_metrics["accuracy"]:>10.4f} {best_metrics["macro_f1"]:>10.4f} '
      f'{best_metrics["weighted_f1"]:>12.4f} {best_metrics["kappa"]:>8.4f}')
print(f'\n  Cascade advantage: +{(best_metrics["macro_f1"] - xgb_f1m)*100:.1f} pp macro F1 over flat XGB')

## 7. Bootstrap Confidence Intervals

In [ ]:
print('Computing bootstrap CIs (1000 iterations)...')
y_true_arr = np.array(y_class9)
y_pred_arr = np.array(best_pred)

In [ ]:
ci_results = {}
for name, fn in [('Accuracy', accuracy_score),
                  ('Macro F1', lambda y, p: f1_score(y, p, average='macro', zero_division=0)),
                  ('Weighted F1', lambda y, p: f1_score(y, p, average='weighted', zero_division=0)),
                  ('Kappa', cohen_kappa_score)]:
    mean, lo, hi = bootstrap_ci(y_true_arr, y_pred_arr, fn, n_boot=1000)
    ci_results[name] = (mean, lo, hi)
    print(f'  {name:15s}: {mean:.4f} [{lo:.4f} – {hi:.4f}]')

══════════════════════════════════════════════════════════════
8. CALIBRATION ANALYSIS
══════════════════════════════════════════════════════════════

## 8. Calibration Analysis
ECE, MCE, and Brier Skill Score at each cascade level.
Isotonic regression as post-hoc sensitivity check.

In [ ]:
# L1 calibration
l1_cal = calibrate_oof_isotonic(l1_proba, y_binary, fold_indices)
l1_ece_raw, _ = compute_ece(y_binary, l1_proba)
l1_ece_cal, _ = compute_ece(y_binary, l1_cal)
l1_bss_raw, _ = scaled_brier_score(y_binary, l1_proba)

In [ ]:
# L2 calibration
l2_proba_best = oof_prob_l2_opt
l2_ece, _ = compute_ece(y_l2, l2_proba_best)
l2_bss, _ = scaled_brier_score(y_l2, l2_proba_best)

In [ ]:
# L3 calibration
l3_proba_raw = oof_p_l3_opt
l3_cal = calibrate_oof_isotonic(l3_proba_raw, y_l3, fold_idx_pos)
l3_ece_raw, _ = compute_ece(y_l3, l3_proba_raw)
l3_ece_cal, _ = compute_ece(y_l3, l3_cal)
l3_bss_raw, _ = scaled_brier_score(y_l3, l3_proba_raw)

In [ ]:
print('=== CALIBRATION SUMMARY ===')
print(f'  L1 ECE: {l1_ece_raw:.4f} → {l1_ece_cal:.4f} (post-isotonic)')
print(f'  L2 ECE: {l2_ece:.4f}')
print(f'  L3 ECE: {l3_ece_raw:.4f} → {l3_ece_cal:.4f} (post-isotonic)')
print(f'  L1 BSS: {l1_bss_raw:.4f} | L2 BSS: {l2_bss:.4f} | L3 BSS: {l3_bss_raw:.4f}')

══════════════════════════════════════════════════════════════
9. CONFORMAL PREDICTION & CONFIDENCE ZONES
══════════════════════════════════════════════════════════════

## 9. Conformal Prediction & Confidence-Based Triaging

In [ ]:
# Build 9-class compound probabilities
cascade_proba_9 = build_cascade_proba_9class(
    l1_proba, l2_proba_best, l3_proba_raw, L1_THRESHOLD, pos_idx)

In [ ]:
# Conformal prediction at multiple significance levels
conformal_results = {}
for alpha in [0.01, 0.05, 0.10, 0.15, 0.20]:
    cr = conformal_prediction(y_class9, cascade_proba_9, alpha=alpha)
    conformal_results[alpha] = cr
    print(f'  α={alpha:.2f}: coverage={cr["coverage"]:.3f}, '
          f'mean_set={cr["mean_set_size"]:.2f}, median_set={cr["median_set_size"]:.0f}')

In [ ]:
# Compound calibration validation
compound_report = validate_compound_calibration(
    cascade_proba_9, y_class9, CLASS9_NAMES, n_bins=10, strategy='quantile')
print(f'\nCompound ECE: {compound_report["ece_overall"]:.4f}')
print(f'Compound BSS: {compound_report["scaled_brier"]:.4f}')

In [ ]:
# L2 ↔ L3 error independence
l2_errors = (oof_pred_l2_opt != y_l2)
l3_errors = ((oof_p_l3_opt >= 0.5).astype(int) != y_l3)
indep = validate_level_independence(l2_errors, l3_errors)
print(f'\nL2↔L3 independence: φ={indep["phi_coefficient"]:.3f}, '
      f'p={indep["p_value"]:.4f} → {indep["interpretation"]}')

In [ ]:
# Confidence zones
CONF_HIGH, CONF_LOW = 0.7, 0.3
conf_df = compute_cascade_confidence(
    l1_proba, l2_proba_best, l3_proba_raw, L1_THRESHOLD, pos_idx)
conf_df['true_class'] = y_class9
conf_df['pred_class'] = best_pred
conf_df['correct'] = (conf_df['true_class'] == conf_df['pred_class']).astype(int)
conf_df['zone'] = conf_df['cascade_conf'].apply(
    lambda c: 'HIGH' if c >= CONF_HIGH else ('MEDIUM' if c >= CONF_LOW else 'LOW'))

In [ ]:
zone_stats = conf_df.groupby('zone').agg(
    n_samples=('correct', 'count'), accuracy=('correct', 'mean'),
    mean_conf=('cascade_conf', 'mean')).reindex(['HIGH', 'MEDIUM', 'LOW'])
print(f'\n=== CONFIDENCE ZONES ===')
print(zone_stats.to_string())

══════════════════════════════════════════════════════════════
10. ERROR ATTRIBUTION
══════════════════════════════════════════════════════════════

## 10. Error Attribution

In [ ]:
error_df = attribute_errors(
    y_true_9=y_class9, cascade_pred=best_pred,
    l1_proba=l1_proba, l1_threshold=L1_THRESHOLD,
    l2_pred_full=oof_pred_l2_opt, l3_pred_full=(oof_p_l3_opt >= 0.5).astype(int),
    y_binary_enc=y_binary, y_heavy=y_l2, y_light=y_l3, pos_idx=pos_idx)

In [ ]:
print(f'Total errors: {len(error_df)}')
print(error_df['error_type'].value_counts().to_string())

══════════════════════════════════════════════════════════════
11. EXTERNAL VALIDATION
══════════════════════════════════════════════════════════════

## 11. External Validation (Frozen Weights)

In [ ]:
print('Extracting external peak features...')
X_ext_3d = D['X_ext_3d']
df_ext_peak = extract_all_features(X_ext_3d, CHANNELS, verbose=True)
df_ext_peak = df_ext_peak.replace([np.inf, -np.inf], 0).fillna(0)
X_ext_peak = df_ext_peak.values.astype(np.float32)

In [ ]:
ext_pred, ext_l1_proba, ext_l2_proba, ext_l3_proba = run_external_cascade(
    l1_opt_models, l2_opt_models, l3_opt_models,
    X_ext_peak, l1_threshold=L1_THRESHOLD)

In [ ]:
y_ext_true = np.array(D['y_ext_class9'])
ext_metrics = compute_cascade_metrics(y_ext_true, ext_pred)

In [ ]:
print(f'\n=== EXTERNAL VALIDATION ===')
print(f'  Accuracy:    {ext_metrics["accuracy"]:.4f}')
print(f'  Macro F1:    {ext_metrics["macro_f1"]:.4f}')
print(f'  Weighted F1: {ext_metrics["weighted_f1"]:.4f}')
print(f'  Cohen κ:     {ext_metrics["kappa"]:.4f}')
print(f'  OOF→Ext gap: {abs(best_metrics["accuracy"] - ext_metrics["accuracy"])*100:.1f} pp')

In [ ]:
# External confidence zones
ext_conf_df = compute_cascade_confidence(
    ext_l1_proba, ext_l2_proba,
    ext_l3_proba, L1_THRESHOLD,
    np.where(ext_l1_proba >= L1_THRESHOLD)[0])
ext_conf_df['pred'] = ext_pred
ext_conf_df['true'] = y_ext_true
ext_conf_df['correct'] = (ext_conf_df['pred'] == ext_conf_df['true']).astype(int)
ext_conf_df['zone'] = ext_conf_df['cascade_conf'].apply(
    lambda c: 'HIGH' if c >= CONF_HIGH else ('MEDIUM' if c >= CONF_LOW else 'LOW'))
ext_zone = ext_conf_df.groupby('zone').agg(
    n=('correct', 'count'), accuracy=('correct', 'mean')).reindex(['HIGH', 'MEDIUM', 'LOW'])
print(f'\nExternal confidence zones:')
print(ext_zone.to_string())

In [ ]:
# Save external results
save_pickle({
    'ext_pred': ext_pred, 'ext_l1_proba': ext_l1_proba,
    'ext_l2_proba': ext_l2_proba, 'ext_l3_proba': ext_l3_proba,
    'ext_metrics': ext_metrics,
}, RESULTS / 'L4_ext_validation_results.pkl')

══════════════════════════════════════════════════════════════
12. SHAP EXPLAINABILITY
══════════════════════════════════════════════════════════════

## 12. SHAP Explainability (TreeSHAP)

In [ ]:
import shap

In [ ]:
N_SHAP = min(500, N)
rng = np.random.RandomState(SEED)
shap_idx = rng.choice(N, N_SHAP, replace=False)

In [ ]:
print(f'Computing TreeSHAP for {N_SHAP} samples...')
all_shap_l1 = np.zeros((N_SHAP, len(peak_cols)))
for i, si in enumerate(shap_idx):
    fold_id = int(fold_indices[si])
    exp = shap.TreeExplainer(l1_opt_models[fold_id])
    sv = exp.shap_values(X_peak[si:si + 1])
    if isinstance(sv, list): sv = sv[1]
    all_shap_l1[i] = sv[0]

In [ ]:
region_shap = aggregate_shap_by_region(all_shap_l1, peak_cols)
print('\nRegion × Channel SHAP heatmap (mean |SHAP|):')
print(region_shap.round(4).to_string())

══════════════════════════════════════════════════════════════
13. CDS PIPELINE DEMO
══════════════════════════════════════════════════════════════

## 13. CDS Pipeline Demo

In [ ]:
CDS_CONFIG = {
    'l1_threshold': L1_THRESHOLD,
    'conf_high': CONF_HIGH,
    'conf_low': CONF_LOW,
}

In [ ]:
demo_classes = ['NEGATIVE', 'IGG_KAPPA', 'IGA_LAMBDA', 'FREE_KAPPA', 'IGM_KAPPA']
print('=' * 65)
print('CDS PIPELINE DEMO — representative samples')
print('=' * 65)
for cls in demo_classes:
    idx = np.where(np.array(y_class9) == cls)[0]
    if len(idx) > 0:
        result = cds_predict(X_peak[idx[0]], l1_opt_models, l2_opt_models,
                             l3_opt_models, CDS_CONFIG)
        match = '✓' if result.prediction == cls else '✗'
        print(f'  {match} True: {cls:16s} → Pred: {result.prediction:16s} '
              f'Zone: {result.zone:6s} Conf: {result.cascade_conf:.3f}')

══════════════════════════════════════════════════════════════
14. SAVE ALL RESULTS
══════════════════════════════════════════════════════════════

## 14. Save All Results

In [ ]:
# Cascade results
save_pickle({
    'best_config': f'{L1_MODEL_KEY} → XGB-Peak-Optuna → XGB-Peak-Optuna',
    'best_threshold': L1_THRESHOLD,
    'best_metrics': {k: v for k, v in best_metrics.items() if k != 'confusion'},
    'best_pred': best_pred,
    'ci_results': ci_results,
    'conf_high': CONF_HIGH, 'conf_low': CONF_LOW,
}, RESULTS / 'L4_cascade_results.pkl')

In [ ]:
# Calibration & validation
save_pickle({
    'l1_calibration': {'ece_raw': l1_ece_raw, 'ece_cal': l1_ece_cal, 'bss': l1_bss_raw},
    'l2_calibration': {'ece': l2_ece, 'bss': l2_bss},
    'l3_calibration': {'ece_raw': l3_ece_raw, 'ece_cal': l3_ece_cal, 'bss': l3_bss_raw},
    'compound_report': compound_report,
    'independence_test': indep,
    'conformal_results': {k: {kk: vv for kk, vv in v.items()
                               if kk not in ('pred_sets', 'test_idx', 'cal_idx')}
                          for k, v in conformal_results.items()},
    'zone_stats': zone_stats.to_dict(),
    'error_attribution': error_df.to_dict(),
    'shap_region_heatmap': region_shap.to_dict(),
}, RESULTS / 'L4_calibration_validation.pkl')

In [ ]:
# Optuna parameters (for Supplemental Table S6)
save_pickle({
    'L1': bp_l1, 'L2': bp_l2, 'L3': bp_l3,
}, RESULTS / 'optuna_best_params.pkl')

In [ ]:
print('\n' + '=' * 70)
print('CORE PIPELINE COMPLETE (§0–§14)')
print('=' * 70)
print(f'\nL1 (Binary):     AUC = {np.mean(cv_xpk_opt["auc_roc"]):.4f}')
print(f'L2 (Heavy Chain): Macro F1 = {np.mean(cv_l2_opt["macro_f1"]):.4f}')
print(f'L3 (Light Chain): AUC = {np.mean(cv_l3_opt["auc_roc"]):.4f}')
print(f'Cascade (9-class): Acc = {best_metrics["accuracy"]:.4f}, '
      f'Macro F1 = {best_metrics["macro_f1"]:.4f}')
print(f'External Val:      Acc = {ext_metrics["accuracy"]:.4f}, '
      f'Macro F1 = {ext_metrics["macro_f1"]:.4f}')
print(f'\nContinuing to §15–§18 (learning curves, tuning-free, nested CV, full SHAP)...')

══════════════════════════════════════════════════════════════
QUICK RELOAD (run only if kernel was restarted before §15)
══════════════════════════════════════════════════════════════

## Quick Reload — Restore Variables from Saved PKL Files
⚠️ **Run this cell only if the kernel was restarted** (e.g., after switching
to High-RAM runtime). If §0–§14 ran in the same session, skip this cell.

In [ ]:
QUICK_RELOAD = False  # ← Set True if kernel was restarted

In [ ]:
if QUICK_RELOAD:
    import numpy as np, pandas as pd, pickle, sys, warnings, time
    from pathlib import Path
    from sklearn.model_selection import learning_curve, PredefinedSplit
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, cohen_kappa_score
    import xgboost as xgb
    warnings.filterwarnings('ignore')

    BASE = Path('/content/drive/MyDrive/001_IT_ML_github')
    sys.path.insert(0, str(BASE))

    from src.constants import *
    from src.evaluation import *
    from src.cascade import *
    from src.calibration import *
    from src.confidence import *
    from src.features import extract_all_features
    from src.utils import Timer, save_pickle, load_pickle

    PROCESSED = BASE / 'data' / 'processed'
    MODELS = BASE / 'models'
    RESULTS = BASE / 'results'

    D = load_pickle(PROCESSED / 'dataset.pkl')
    PF = load_pickle(PROCESSED / 'peak_features.pkl')
    FA = load_pickle(RESULTS / 'fold_assignments.pkl')
    L1 = load_pickle(RESULTS / 'L1_oof_predictions.pkl')
    L2 = load_pickle(RESULTS / 'L2_oof_predictions.pkl')
    L3 = load_pickle(RESULTS / 'L3_oof_predictions.pkl')
    L4 = load_pickle(RESULTS / 'L4_cascade_results.pkl')
    EXT = load_pickle(RESULTS / 'L4_ext_validation_results.pkl')
    OPTUNA = load_pickle(RESULTS / 'optuna_best_params.pkl')

    X_3d = D['X_3d']
    X_peak = PF['X_peak']
    peak_cols = PF['columns']
    fold_indices = FA['fold_indices']
    y_binary = L1['y_true']
    y_class9 = np.array(D['y_class9'])
    N = len(y_binary)

    le_9class = LabelEncoder(); le_9class.fit(y_class9)
    y_9enc = le_9class.transform(y_class9)

    pos_mask = (y_binary == 1)
    pos_idx = np.where(pos_mask)[0]
    N_pos = len(pos_idx)

    X_l2 = X_peak[pos_idx]
    X_3d_l2 = X_3d[pos_idx]
    fold_idx_pos = fold_indices[pos_idx]

    le_heavy = LabelEncoder(); le_heavy.classes_ = np.array(L2_CLASSES)
    y_l2 = le_heavy.transform(D['y_heavy'][pos_idx])
    le_light = LabelEncoder(); le_light.classes_ = np.array(L3_CLASSES)
    y_l3 = le_light.transform(D['y_light'][pos_idx])

    bp_l1 = OPTUNA['L1']; bp_l2 = OPTUNA['L2']; bp_l3 = OPTUNA['L3']

    l1_opt_models = get_fold_models(load_pickle(MODELS / 'L1_xgb_peak_optuna_models.pkl'))
    l2_opt_models = get_fold_models(load_pickle(MODELS / 'L2_xgb_peak_optuna_models.pkl'))
    l3_opt_models = get_fold_models(load_pickle(MODELS / 'L3_xgb_peak_optuna_models.pkl'))

    L1_MODEL_KEY = [k for k in L1['oof_proba'] if 'Optuna' in k or 'optuna' in k][0]
    l1_proba = L1['oof_proba'][L1_MODEL_KEY]
    L1_THRESHOLD = L1['threshold_analysis'][L1_MODEL_KEY]['youden']['threshold']
    best_pred = L4['best_pred']
    best_metrics = L4.get('best_metrics', {})
    if not best_metrics:
        best_metrics = compute_cascade_metrics(y_class9, best_pred)

    ext_pred = EXT['ext_pred']
    y_ext_true = np.array(D['y_ext_class9'])
    X_ext_3d = D['X_ext_3d']
    ext_l1_proba = EXT['ext_l1_proba']
    ext_metrics = EXT.get('ext_metrics', compute_cascade_metrics(y_ext_true, ext_pred))

    ext_peak_path = PROCESSED / 'ext_peak_features.pkl'
    if ext_peak_path.exists():
        X_ext_peak = load_pickle(ext_peak_path)['X_peak']
    else:
        df_ext = extract_all_features(X_ext_3d, CHANNELS, verbose=True)
        df_ext = df_ext.replace([np.inf, -np.inf], 0).fillna(0)
        X_ext_peak = df_ext.values.astype(np.float32)
        save_pickle({'X_peak': X_ext_peak, 'columns': list(df_ext.columns)}, ext_peak_path)

    # L2/L3 proba for SHAP
    l2_proba_best = L2['oof_prob']['XGBoost-Peak-Optuna']
    l3_key = 'XGBoost-Peak-Optuna'
    if 'oof_proba' in L3 and l3_key in L3['oof_proba']:
        l3_proba_raw = L3['oof_proba'][l3_key]
    elif 'oof_prob' in L3 and l3_key in L3['oof_prob']:
        l3_full = L3['oof_prob'][l3_key]
        l3_proba_raw = l3_full[:, 1] if l3_full.ndim == 2 else l3_full
    oof_prob_l2_opt = l2_proba_best

    print(f'✓ Quick reload complete')
    print(f'  N={N}, N_pos={N_pos}, L1_thr={L1_THRESHOLD:.4f}')
    print(f'  Cascade acc: {best_metrics.get("accuracy", "?"):.4f}')
else:
    print('Quick reload skipped (same session)')

══════════════════════════════════════════════════════════════
15. LEARNING CURVES
══════════════════════════════════════════════════════════════

## 15. Learning Curves (L1 / L2 / L3)
Sample efficiency analysis: train/val scores at 10 subsample points.
Uses the same fold assignments as the main pipeline.
L1: AUC-ROC, L2: Macro F1, L3: AUC-ROC.

In [ ]:
print('=== Learning Curves ===')

In [ ]:
lc_results = {}

In [ ]:
# L1: Binary detection
print('  L1 (AUC-ROC)...')
ps_l1 = PredefinedSplit(fold_indices)
xgb_lc_l1 = xgb.XGBClassifier(
    **bp_l1, eval_metric='logloss', use_label_encoder=False,
    random_state=SEED, early_stopping_rounds=None)

In [ ]:
with Timer() as t:
    sizes_l1, train_l1, val_l1 = learning_curve(
        xgb_lc_l1, X_peak, y_binary, cv=ps_l1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='roc_auc', n_jobs=-1)

In [ ]:
lc_results['L1'] = {
    'train_sizes': sizes_l1, 'train_scores': train_l1,
    'val_scores': val_l1, 'metric': 'roc_auc',
}
print(f'    Final val AUC: {val_l1[-1].mean():.4f} ± {val_l1[-1].std():.4f} ({t.elapsed:.0f}s)')

In [ ]:
# L2: Heavy chain (positive subset)
print('  L2 (Macro F1)...')
ps_l2 = PredefinedSplit(fold_idx_pos)
xgb_lc_l2 = xgb.XGBClassifier(
    **bp_l2, objective='multi:softprob', num_class=4,
    eval_metric='mlogloss', use_label_encoder=False,
    random_state=SEED, early_stopping_rounds=None)

In [ ]:
X_l2 = X_peak[pos_idx]
with Timer() as t:
    sizes_l2, train_l2, val_l2 = learning_curve(
        xgb_lc_l2, X_l2, y_l2, cv=ps_l2,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='f1_macro', n_jobs=-1)

In [ ]:
lc_results['L2'] = {
    'train_sizes': sizes_l2, 'train_scores': train_l2,
    'val_scores': val_l2, 'metric': 'f1_macro',
}
print(f'    Final val F1: {val_l2[-1].mean():.4f} ± {val_l2[-1].std():.4f} ({t.elapsed:.0f}s)')

In [ ]:
# L3: Light chain (positive subset)
print('  L3 (AUC-ROC)...')
xgb_lc_l3 = xgb.XGBClassifier(
    **bp_l3, eval_metric='logloss', use_label_encoder=False,
    random_state=SEED, early_stopping_rounds=None)

In [ ]:
with Timer() as t:
    sizes_l3, train_l3, val_l3 = learning_curve(
        xgb_lc_l3, X_l2, y_l3, cv=ps_l2,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='roc_auc', n_jobs=-1)

In [ ]:
lc_results['L3'] = {
    'train_sizes': sizes_l3, 'train_scores': train_l3,
    'val_scores': val_l3, 'metric': 'roc_auc',
}
print(f'    Final val AUC: {val_l3[-1].mean():.4f} ± {val_l3[-1].std():.4f} ({t.elapsed:.0f}s)')

In [ ]:
save_pickle(lc_results, RESULTS / 'L4_learning_curves.pkl')
print('  Learning curves saved ✓')

══════════════════════════════════════════════════════════════
16. TUNING-FREE CASCADE
══════════════════════════════════════════════════════════════

## 16. Tuning-Free Cascade (Default XGBoost)
Replicates cascade with default hyperparameters (no Optuna) to confirm
negligible hyperparameter overfitting. Same fold assignments and features.

In [ ]:
print('=== Tuning-Free Cascade (Default XGBoost, No Optuna) ===')

In [ ]:
# L1 default
print('  L1 default...')
oof_p_l1_def = np.zeros(N, dtype=np.float32)
l1_def_models = []
with Timer() as t:
    for fn in range(N_FOLDS):
        te = fold_indices == fn
        tri, tei = np.where(~te)[0], np.where(te)[0]
        m = xgb.XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            eval_metric='logloss', use_label_encoder=False,
            random_state=SEED, early_stopping_rounds=50)
        m.fit(X_peak[tri], y_binary[tri],
              eval_set=[(X_peak[tei], y_binary[tei])], verbose=False)
        oof_p_l1_def[tei] = m.predict_proba(X_peak[tei])[:, 1]
        l1_def_models.append(m)
l1_def_thr = find_optimal_thresholds(y_binary, oof_p_l1_def)['youden']['threshold']
print(f'    AUC: {roc_auc_score(y_binary, oof_p_l1_def):.4f}, Thr: {l1_def_thr:.4f} ({t.elapsed:.0f}s)')

In [ ]:
# L2 default
print('  L2 default...')
oof_pred_l2_def = np.zeros(N_pos, dtype=np.int8)
l2_def_models = []
with Timer() as t:
    for fn in range(N_FOLDS):
        te = fold_idx_pos == fn
        tri, tei = np.where(~te)[0], np.where(te)[0]
        m = xgb.XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            objective='multi:softprob', num_class=4,
            eval_metric='mlogloss', use_label_encoder=False,
            random_state=SEED, early_stopping_rounds=50)
        m.fit(X_l2[tri], y_l2[tri],
              eval_set=[(X_l2[tei], y_l2[tei])], verbose=False)
        oof_pred_l2_def[tei] = m.predict(X_l2[tei])
        l2_def_models.append(m)
print(f'    Macro F1: {f1_score(y_l2, oof_pred_l2_def, average="macro"):.4f} ({t.elapsed:.0f}s)')

In [ ]:
# L3 default
print('  L3 default...')
oof_p_l3_def = np.zeros(N_pos, dtype=np.float32)
l3_def_models = []
with Timer() as t:
    for fn in range(N_FOLDS):
        te = fold_idx_pos == fn
        tri, tei = np.where(~te)[0], np.where(te)[0]
        m = xgb.XGBClassifier(
            n_estimators=500, max_depth=6, learning_rate=0.1,
            eval_metric='logloss', use_label_encoder=False,
            random_state=SEED, early_stopping_rounds=50)
        m.fit(X_l2[tri], y_l3[tri],
              eval_set=[(X_l2[tei], y_l3[tei])], verbose=False)
        oof_p_l3_def[tei] = m.predict_proba(X_l2[tei])[:, 1]
        l3_def_models.append(m)
print(f'    AUC: {roc_auc_score(y_l3, oof_p_l3_def):.4f} ({t.elapsed:.0f}s)')

In [ ]:
# Assemble tuning-free cascade
l3_pred_def = (oof_p_l3_def >= 0.5).astype(int)
def_pred, _, _ = assemble_cascade_oof(
    oof_p_l1_def, oof_pred_l2_def, l3_pred_def, l1_def_thr,
    pos_idx, y_binary, l2_def_models, l3_def_models, X_peak)

In [ ]:
def_acc = accuracy_score(y_class9, def_pred)
def_f1m = f1_score(y_class9, def_pred, labels=CLASS9_NAMES, average='macro', zero_division=0)
def_kappa = cohen_kappa_score(y_class9, def_pred)

In [ ]:
# External with default models
from src.cascade import _ensemble_predict
ext_l1_def = _ensemble_predict(l1_def_models, X_ext_peak, task='binary')
ext_pos_def = np.where(ext_l1_def >= l1_def_thr)[0]
ext_pred_def = np.full(len(y_ext_true), 'NEGATIVE', dtype='U20')
if len(ext_pos_def) > 0:
    ext_l2_def = _ensemble_predict(l2_def_models, X_ext_peak[ext_pos_def], task='multi')
    ext_l3_def = _ensemble_predict(l3_def_models, X_ext_peak[ext_pos_def], task='binary')
    for j, idx in enumerate(ext_pos_def):
        h = L2_CLASSES[np.argmax(ext_l2_def[j])]
        l = L3_CLASSES[int(ext_l3_def[j] >= 0.5)]
        ext_pred_def[idx] = f'{h}_{l}'

In [ ]:
ext_def_acc = accuracy_score(y_ext_true, ext_pred_def)

In [ ]:
print(f'\n=== TUNING-FREE vs OPTUNA COMPARISON ===')
print(f'  {"":20s} {"Optuna":>10s} {"Default":>10s} {"Δ":>8s}')
print(f'  {"OOF Accuracy":20s} {best_metrics["accuracy"]:>10.4f} {def_acc:>10.4f} {best_metrics["accuracy"]-def_acc:>+8.4f}')
print(f'  {"OOF Macro F1":20s} {best_metrics["macro_f1"]:>10.4f} {def_f1m:>10.4f} {best_metrics["macro_f1"]-def_f1m:>+8.4f}')
print(f'  {"OOF κ":20s} {best_metrics["kappa"]:>10.4f} {def_kappa:>10.4f} {best_metrics["kappa"]-def_kappa:>+8.4f}')
print(f'  {"Ext Accuracy":20s} {ext_metrics["accuracy"]:>10.4f} {ext_def_acc:>10.4f} {ext_metrics["accuracy"]-ext_def_acc:>+8.4f}')

In [ ]:
save_pickle({
    'default_oof_pred': def_pred, 'default_ext_pred': ext_pred_def,
    'default_metrics': {'accuracy': def_acc, 'macro_f1': def_f1m, 'kappa': def_kappa},
    'default_ext_metrics': {'accuracy': ext_def_acc},
    'l1_threshold': l1_def_thr,
}, RESULTS / 'L4_tuning_free_results.pkl')
save_pickle(l1_def_models, MODELS / 'L1_default_models.pkl')
save_pickle(l2_def_models, MODELS / 'L2_default_models.pkl')
save_pickle(l3_def_models, MODELS / 'L3_default_models.pkl')
print('  Tuning-free results saved ✓')

══════════════════════════════════════════════════════════════
17. NESTED CV — OPTUNA (OPTIONAL)
══════════════════════════════════════════════════════════════

## 17. Nested CV — Optuna (⚠️ OPTIONAL — ~45 min)
Gold-standard evaluation: outer 5-fold CV with per-fold inner Optuna
optimization. Prevents any hyperparameter information leaking into
OOF performance estimates.

**Set `RUN_NESTED_CV = True` to execute.**

In [ ]:
RUN_NESTED_CV = False  # ⚠️ Set True to run (~45 min on Colab High-RAM)

In [ ]:
if RUN_NESTED_CV:
    print('=== Nested CV — Optuna ===')
    print('  5 outer folds × inner Optuna per fold')
    N_INNER_TRIALS = 50  # fewer trials per inner loop (vs 100 in single-loop)

    nested_oof_pred = np.full(N, 'NEGATIVE', dtype='U20')
    nested_l1_proba = np.zeros(N, dtype=np.float32)
    nested_bp_per_fold = {}

    with Timer() as t_nested:
        for outer_fold in range(N_FOLDS):
            print(f'\n  Outer fold {outer_fold + 1}/{N_FOLDS}')
            outer_test = fold_indices == outer_fold
            outer_train = ~outer_test
            outer_tri = np.where(outer_train)[0]
            outer_tei = np.where(outer_test)[0]

            # Inner fold assignments (4-fold on outer train)
            inner_folds = np.zeros(len(outer_tri), dtype=np.int8)
            skf_inner = StratifiedKFold(n_splits=N_FOLDS - 1, shuffle=True,
                                         random_state=SEED + outer_fold)
            for i, (_, tei) in enumerate(skf_inner.split(
                    X_peak[outer_tri], y_binary[outer_tri])):
                inner_folds[tei] = i

            # --- Inner Optuna L1 ---
            def inner_l1_objective(trial):
                params = {
                    'n_estimators':     trial.suggest_int('n_estimators', 200, 1500),
                    'max_depth':        trial.suggest_int('max_depth', 3, 8),
                    'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                    'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
                    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
                    'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
                    'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
                    'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                }
                aucs = []
                for ifn in range(N_FOLDS - 1):
                    ite = inner_folds == ifn
                    itri = np.where(~ite)[0]
                    itei = np.where(ite)[0]
                    m = xgb.XGBClassifier(**params, eval_metric='logloss',
                                          use_label_encoder=False, random_state=SEED,
                                          early_stopping_rounds=30)
                    m.fit(X_peak[outer_tri[itri]], y_binary[outer_tri[itri]],
                          eval_set=[(X_peak[outer_tri[itei]], y_binary[outer_tri[itei]])],
                          verbose=False)
                    prob = m.predict_proba(X_peak[outer_tri[itei]])[:, 1]
                    aucs.append(roc_auc_score(y_binary[outer_tri[itei]], prob))
                return np.mean(aucs)

            study = optuna.create_study(direction='maximize',
                                         sampler=optuna.samplers.TPESampler(seed=SEED + outer_fold))
            study.optimize(inner_l1_objective, n_trials=N_INNER_TRIALS, show_progress_bar=False)
            bp_nested_l1 = study.best_params
            print(f'    L1 inner best AUC: {study.best_value:.4f}')

            # Train L1 on full outer train with inner-best params
            m_l1 = xgb.XGBClassifier(**bp_nested_l1, eval_metric='logloss',
                                      use_label_encoder=False, random_state=SEED,
                                      early_stopping_rounds=50)
            m_l1.fit(X_peak[outer_tri], y_binary[outer_tri],
                     eval_set=[(X_peak[outer_tei], y_binary[outer_tei])], verbose=False)
            l1_prob_outer = m_l1.predict_proba(X_peak[outer_tei])[:, 1]
            nested_l1_proba[outer_tei] = l1_prob_outer

            # --- Inner Optuna L2 (positive subset of outer train) ---
            pos_in_outer_train = np.array([i for i in outer_tri if i in set(pos_idx)])
            pos_in_outer_test  = np.array([i for i in outer_tei if i in set(pos_idx)])

            if len(pos_in_outer_train) > 10 and len(pos_in_outer_test) > 0:
                # Map to L2 indices
                pos_idx_list = list(pos_idx)
                l2_tri = np.array([pos_idx_list.index(i) for i in pos_in_outer_train])
                l2_tei = np.array([pos_idx_list.index(i) for i in pos_in_outer_test])

                inner_folds_l2 = np.zeros(len(l2_tri), dtype=np.int8)
                skf_l2 = StratifiedKFold(n_splits=min(N_FOLDS - 1, 4), shuffle=True,
                                          random_state=SEED + outer_fold)
                for i, (_, tei_l2) in enumerate(skf_l2.split(X_l2[l2_tri], y_l2[l2_tri])):
                    inner_folds_l2[tei_l2] = i

                def inner_l2_objective(trial):
                    params = {
                        'n_estimators':     trial.suggest_int('n_estimators', 200, 1500),
                        'max_depth':        trial.suggest_int('max_depth', 3, 10),
                        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
                        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
                        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
                        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
                        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                    }
                    f1s = []
                    n_inner = len(np.unique(inner_folds_l2))
                    for ifn in range(n_inner):
                        ite = inner_folds_l2 == ifn
                        m = xgb.XGBClassifier(**params, objective='multi:softprob', num_class=4,
                                              eval_metric='mlogloss', use_label_encoder=False,
                                              random_state=SEED, early_stopping_rounds=30)
                        m.fit(X_l2[l2_tri[~ite]], y_l2[l2_tri[~ite]],
                              eval_set=[(X_l2[l2_tri[ite]], y_l2[l2_tri[ite]])], verbose=False)
                        f1s.append(f1_score(y_l2[l2_tri[ite]], m.predict(X_l2[l2_tri[ite]]),
                                            average='macro'))
                    return np.mean(f1s)

                study_l2n = optuna.create_study(direction='maximize',
                                                 sampler=optuna.samplers.TPESampler(seed=SEED + outer_fold))
                study_l2n.optimize(inner_l2_objective, n_trials=N_INNER_TRIALS, show_progress_bar=False)

                m_l2 = xgb.XGBClassifier(**study_l2n.best_params, objective='multi:softprob', num_class=4,
                                          eval_metric='mlogloss', use_label_encoder=False,
                                          random_state=SEED, early_stopping_rounds=50)
                m_l2.fit(X_l2[l2_tri], y_l2[l2_tri],
                         eval_set=[(X_l2[l2_tei], y_l2[l2_tei])], verbose=False)

                # L3 (simpler — use same inner structure)
                def inner_l3_objective(trial):
                    params = {
                        'n_estimators':     trial.suggest_int('n_estimators', 200, 1500),
                        'max_depth':        trial.suggest_int('max_depth', 3, 8),
                        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
                        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
                        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
                        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-3, 10, log=True),
                        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
                        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                    }
                    aucs = []
                    n_inner = len(np.unique(inner_folds_l2))
                    for ifn in range(n_inner):
                        ite = inner_folds_l2 == ifn
                        m = xgb.XGBClassifier(**params, eval_metric='logloss',
                                              use_label_encoder=False, random_state=SEED,
                                              early_stopping_rounds=30)
                        m.fit(X_l2[l2_tri[~ite]], y_l3[l2_tri[~ite]],
                              eval_set=[(X_l2[l2_tri[ite]], y_l3[l2_tri[ite]])], verbose=False)
                        aucs.append(roc_auc_score(y_l3[l2_tri[ite]],
                                                   m.predict_proba(X_l2[l2_tri[ite]])[:, 1]))
                    return np.mean(aucs)

                study_l3n = optuna.create_study(direction='maximize',
                                                 sampler=optuna.samplers.TPESampler(seed=SEED + outer_fold))
                study_l3n.optimize(inner_l3_objective, n_trials=N_INNER_TRIALS, show_progress_bar=False)

                m_l3 = xgb.XGBClassifier(**study_l3n.best_params, eval_metric='logloss',
                                          use_label_encoder=False, random_state=SEED,
                                          early_stopping_rounds=50)
                m_l3.fit(X_l2[l2_tri], y_l3[l2_tri],
                         eval_set=[(X_l2[l2_tei], y_l3[l2_tei])], verbose=False)

                # Predict outer test positives
                l1_pos = l1_prob_outer >= find_optimal_thresholds(
                    y_binary[outer_tri], m_l1.predict_proba(X_peak[outer_tri])[:, 1]
                )['youden']['threshold']

                for j, idx in enumerate(outer_tei):
                    if l1_pos[j] and idx in set(pos_idx):
                        oi = pos_idx_list.index(idx)
                        l2p = m_l2.predict(X_l2[oi:oi+1])[0]
                        l3p = m_l3.predict_proba(X_l2[oi:oi+1])[0, 1]
                        h = L2_CLASSES[l2p]
                        l = L3_CLASSES[int(l3p >= 0.5)]
                        nested_oof_pred[idx] = f'{h}_{l}'
                    elif l1_pos[j]:
                        # FP — infer with outer-fold models
                        l2p = m_l2.predict_proba(X_peak[idx:idx+1])[0]
                        l3p = m_l3.predict_proba(X_peak[idx:idx+1])[0, 1]
                        h = L2_CLASSES[np.argmax(l2p)]
                        l = L3_CLASSES[int(l3p >= 0.5)]
                        nested_oof_pred[idx] = f'{h}_{l}'

            nested_bp_per_fold[outer_fold] = {
                'L1': bp_nested_l1,
                'L2': study_l2n.best_params if 'study_l2n' in dir() else {},
                'L3': study_l3n.best_params if 'study_l3n' in dir() else {},
            }

    nested_acc = accuracy_score(y_class9, nested_oof_pred)
    nested_f1m = f1_score(y_class9, nested_oof_pred, labels=CLASS9_NAMES,
                           average='macro', zero_division=0)
    nested_kappa = cohen_kappa_score(y_class9, nested_oof_pred)

    print(f'\n=== NESTED CV vs SINGLE-LOOP COMPARISON ===')
    print(f'  {"":20s} {"Single-Loop":>12s} {"Nested CV":>12s} {"Δ":>8s}')
    print(f'  {"Accuracy":20s} {best_metrics["accuracy"]:>12.4f} {nested_acc:>12.4f} {best_metrics["accuracy"]-nested_acc:>+8.4f}')
    print(f'  {"Macro F1":20s} {best_metrics["macro_f1"]:>12.4f} {nested_f1m:>12.4f} {best_metrics["macro_f1"]-nested_f1m:>+8.4f}')
    print(f'  {"κ":20s} {best_metrics["kappa"]:>12.4f} {nested_kappa:>12.4f} {best_metrics["kappa"]-nested_kappa:>+8.4f}')
    print(f'  Total time: {t_nested.elapsed:.0f}s')

    save_pickle({
        'nested_pred': nested_oof_pred,
        'nested_metrics': {'accuracy': nested_acc, 'macro_f1': nested_f1m, 'kappa': nested_kappa},
        'single_loop_metrics': {k: v for k, v in best_metrics.items() if k != 'confusion'},
        'bp_per_fold': nested_bp_per_fold,
        'n_inner_trials': N_INNER_TRIALS,
    }, RESULTS / 'L4_nested_cv_results.pkl')
    print('  Nested CV results saved ✓')
else:
    print('§17 Nested CV — SKIPPED (set RUN_NESTED_CV = True to run)')

══════════════════════════════════════════════════════════════
18. FULL SHAP COMPUTATION
══════════════════════════════════════════════════════════════

## 18. Full SHAP Computation (All Samples × All Levels × Both Cohorts)
TreeSHAP exact values (tree_path_dependent, deterministic) for every sample.
Internal: fold-aware (each sample uses the model that did NOT train on it).
External: 5-fold average SHAP (all models equally valid).

Split into 4 cells so partial progress is preserved if a cell fails.

### 18a. Setup + L1 Internal

In [ ]:
import shap

In [ ]:
# Ensure L2 proba is available
l2_proba_best = L2['oof_prob']['XGBoost-Peak-Optuna']
feat_names = list(peak_cols)

In [ ]:
shap_cache = RESULTS / 'shap_all_levels_all_samples.pkl'

In [ ]:
if shap_cache.exists():
    print('SHAP cache already exists — loading...')
    SHAP = load_pickle(shap_cache)
    shap_l1_int = SHAP['l1_internal']
    shap_l1_ext = SHAP['l1_external']
    shap_l2_int = SHAP['l2_internal']
    shap_l2_ext = SHAP['l2_external']
    shap_l3_int = SHAP['l3_internal']
    shap_l3_ext = SHAP['l3_external']
    shap_l1_base = SHAP.get('l1_base_internal', None)
    shap_l1_base_ext = SHAP.get('l1_base_external', None)
    ext_pos_idx_arr = SHAP.get('ext_pos_idx', np.where(ext_l1_proba >= L1_THRESHOLD)[0])
    print(f'  L1 int: {shap_l1_int.shape}, ext: {shap_l1_ext.shape}')
    print(f'  L2 int: {shap_l2_int.shape}, ext: {shap_l2_ext.shape}')
    print(f'  L3 int: {shap_l3_int.shape}, ext: {shap_l3_ext.shape}')
    print('  Skipping computation — delete cache to recompute.')
else:
    print('=== §18a: L1 Internal SHAP ===')
    shap_l1_int = np.zeros((N, len(feat_names)), dtype=np.float32)
    shap_l1_base = np.zeros(N, dtype=np.float32)
    with Timer() as t:
        for i in range(N):
            if i % 500 == 0 and i > 0: print(f'    {i}/{N}')
            fold_id = int(fold_indices[i])
            exp = shap.TreeExplainer(l1_opt_models[fold_id])
            sv = exp.shap_values(X_peak[i:i+1])
            if isinstance(sv, list): sv = sv[1]
            shap_l1_int[i] = sv[0]
            ev = exp.expected_value
            shap_l1_base[i] = ev[1] if isinstance(ev, (list, np.ndarray)) else ev
    print(f'  L1 Internal done: {shap_l1_int.shape} ({t.elapsed:.0f}s)')

### 18b. L1 External + L2 Internal + L2 External

In [ ]:
if not shap_cache.exists():
    # L1 External (5-fold average)
    N_ext = len(y_ext_true)
    print(f'=== §18b: L1 External ({N_ext} samples, 5-fold avg) ===')
    shap_l1_ext = np.zeros((N_ext, len(feat_names)), dtype=np.float32)
    shap_l1_base_ext = np.zeros(N_ext, dtype=np.float32)
    with Timer() as t:
        for i in range(N_ext):
            if i % 200 == 0 and i > 0: print(f'    {i}/{N_ext}')
            sv_sum = np.zeros(len(feat_names), dtype=np.float64)
            base_sum = 0.0
            for fold_m in l1_opt_models:
                exp = shap.TreeExplainer(fold_m)
                sv = exp.shap_values(X_ext_peak[i:i+1])
                if isinstance(sv, list): sv = sv[1]
                sv_sum += sv[0]
                ev = exp.expected_value
                base_sum += (ev[1] if isinstance(ev, (list, np.ndarray)) else ev)
            shap_l1_ext[i] = sv_sum / N_FOLDS
            shap_l1_base_ext[i] = base_sum / N_FOLDS
    print(f'  L1 External done: {shap_l1_ext.shape} ({t.elapsed:.0f}s)')

    # L2 Internal (fold-aware, predicted-class SHAP)
    print(f'\n  L2 Internal ({N_pos} samples)...')
    shap_l2_int = np.zeros((N_pos, len(feat_names)), dtype=np.float32)
    with Timer() as t:
        for i in range(N_pos):
            if i % 300 == 0 and i > 0: print(f'    {i}/{N_pos}')
            fold_id = int(fold_idx_pos[i])
            exp = shap.TreeExplainer(l2_opt_models[fold_id])
            sv = exp.shap_values(X_l2[i:i+1])
            pred_cls = int(np.argmax(l2_proba_best[i]))
            if isinstance(sv, list):
                shap_l2_int[i] = sv[pred_cls][0]
            elif sv.ndim == 3:
                shap_l2_int[i] = sv[0, :, pred_cls]
            else:
                shap_l2_int[i] = sv[0]
    print(f'  L2 Internal done: {shap_l2_int.shape} ({t.elapsed:.0f}s)')

    # L2 External (5-fold average)
    ext_pos_mask = ext_l1_proba >= L1_THRESHOLD
    ext_pos_idx_arr = np.where(ext_pos_mask)[0]
    N_ext_pos = len(ext_pos_idx_arr)
    print(f'\n  L2 External ({N_ext_pos} positive samples, 5-fold avg)...')
    shap_l2_ext = np.zeros((N_ext_pos, len(feat_names)), dtype=np.float32)
    with Timer() as t:
        for j, idx in enumerate(ext_pos_idx_arr):
            if j % 100 == 0 and j > 0: print(f'    {j}/{N_ext_pos}')
            x = X_ext_peak[idx:idx+1]
            l2p_avg = np.mean([m.predict_proba(x)[0] for m in l2_opt_models], axis=0)
            pred_cls = int(np.argmax(l2p_avg))
            sv_sum = np.zeros(len(feat_names), dtype=np.float64)
            for fold_m in l2_opt_models:
                exp = shap.TreeExplainer(fold_m)
                sv = exp.shap_values(x)
                if isinstance(sv, list):
                    sv_sum += sv[pred_cls][0]
                elif sv.ndim == 3:
                    sv_sum += sv[0, :, pred_cls]
                else:
                    sv_sum += sv[0]
            shap_l2_ext[j] = sv_sum / N_FOLDS
    print(f'  L2 External done: {shap_l2_ext.shape} ({t.elapsed:.0f}s)')

### 18c. L3 Internal + L3 External

In [ ]:
if not shap_cache.exists():
    # L3 Internal (fold-aware)
    print(f'=== §18c: L3 Internal ({N_pos} samples) ===')
    shap_l3_int = np.zeros((N_pos, len(feat_names)), dtype=np.float32)
    with Timer() as t:
        for i in range(N_pos):
            if i % 300 == 0 and i > 0: print(f'    {i}/{N_pos}')
            fold_id = int(fold_idx_pos[i])
            exp = shap.TreeExplainer(l3_opt_models[fold_id])
            sv = exp.shap_values(X_l2[i:i+1])
            if isinstance(sv, list): sv = sv[1]
            shap_l3_int[i] = sv[0]
    print(f'  L3 Internal done: {shap_l3_int.shape} ({t.elapsed:.0f}s)')

    # L3 External (5-fold average)
    print(f'\n  L3 External ({N_ext_pos} samples, 5-fold avg)...')
    shap_l3_ext = np.zeros((N_ext_pos, len(feat_names)), dtype=np.float32)
    with Timer() as t:
        for j, idx in enumerate(ext_pos_idx_arr):
            if j % 100 == 0 and j > 0: print(f'    {j}/{N_ext_pos}')
            x = X_ext_peak[idx:idx+1]
            sv_sum = np.zeros(len(feat_names), dtype=np.float64)
            for fold_m in l3_opt_models:
                exp = shap.TreeExplainer(fold_m)
                sv = exp.shap_values(x)
                if isinstance(sv, list): sv = sv[1]
                sv_sum += sv[0]
            shap_l3_ext[j] = sv_sum / N_FOLDS
    print(f'  L3 External done: {shap_l3_ext.shape} ({t.elapsed:.0f}s)')

### 18d. Save All SHAP Values

In [ ]:
if not shap_cache.exists():
    SHAP = {
        'l1_internal': shap_l1_int, 'l1_external': shap_l1_ext,
        'l2_internal': shap_l2_int, 'l2_external': shap_l2_ext,
        'l3_internal': shap_l3_int, 'l3_external': shap_l3_ext,
        'l1_base_internal': shap_l1_base,
        'l1_base_external': shap_l1_base_ext,
        'ext_pos_idx': ext_pos_idx_arr,
        'feature_names': feat_names,
    }
    save_pickle(SHAP, shap_cache)

In [ ]:
print('=== SHAP Summary ===')
for k in ['l1_internal', 'l1_external', 'l2_internal', 'l2_external',
          'l3_internal', 'l3_external']:
    arr = SHAP.get(k) if 'SHAP' in dir() else eval(f'shap_{k.split("_")[0]}_{k.split("_")[1][:3]}')
    if hasattr(arr, 'shape'):
        print(f'  {k}: {arr.shape}')
print('All SHAP values saved ✓')

══════════════════════════════════════════════════════════════
FINAL SUMMARY
══════════════════════════════════════════════════════════════

## Final Summary

In [ ]:
print('\n' + '=' * 70)
print('FULL PIPELINE COMPLETE (§0–§18)')
print('=' * 70)
print(f'\n--- Core Results ---')
print(f'  Cascade (9-class): Acc = {best_metrics["accuracy"]:.4f}, '
      f'Macro F1 = {best_metrics["macro_f1"]:.4f}, κ = {best_metrics["kappa"]:.4f}')
print(f'  External Val:      Acc = {ext_metrics["accuracy"]:.4f}')
print(f'  Tuning-free:       Acc = {def_acc:.4f} (OOF), {ext_def_acc:.4f} (Ext)')
print(f'\n--- Saved Outputs ---')
for f in sorted(RESULTS.glob('*.pkl')):
    print(f'  {f.name} ({f.stat().st_size/1024:.0f} KB)')
for f in sorted(MODELS.glob('*.pkl')):
    print(f'  {f.name} ({f.stat().st_size/1024:.0f} KB)')
print(f'\nAll results: {RESULTS}')
print(f'All models:  {MODELS}')